In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


In [5]:
##_______________________________________________________________ "" ____________________________________________________________

In [6]:
# ============================================================
# SEPTEMBER 2026 — MESSAGE PLAN
# REBUILD FROM ZERO
#
# SOURCE OF TRUTH: queue
# ============================================================

import pandas as pd
import numpy as np

SEP01 = pd.Timestamp("2026-09-01")
SEP30 = pd.Timestamp("2026-09-30")

# ============================================================
# 1. START FROM QUEUE
# ============================================================

base = queue.copy()

base["customer_id"] = base["customer_id"].astype(str)

base["in_collections_since"] = pd.to_datetime(
    base["in_collections_since"]
)

# ============================================================
# 2. BASIC QA
# ============================================================

print("=" * 90)
print("STEP 1 — SEPTEMBER QUEUE")
print("=" * 90)

print(f"Rows                 : {len(base):,}")
print(f"Unique customers     : {base['customer_id'].nunique():,}")
print(f"Duplicated customers : {base['customer_id'].duplicated().sum():,}")

print(
    f"Outstanding balance  : "
    f"R$ {base['outstanding_balance_brl'].sum():,.2f}"
)

print("\nIN COLLECTIONS SINCE")
print(
    base["in_collections_since"]
    .agg(["min", "max"])
)

print("\nDPD ON 01/09")
print(
    base["days_past_due_on_2026-09-01"]
    .describe()
)

# ============================================================
# 3. HARD ASSERTIONS
# ============================================================

assert len(base) == 10_658, \
    f"Expected 10,658 rows, got {len(base):,}"

assert base["customer_id"].nunique() == 10_658, \
    "customer_id is not unique"

assert base["customer_id"].duplicated().sum() == 0

print("\n✓ Queue reconciled: 10,658 unique customers")

STEP 1 — SEPTEMBER QUEUE
Rows                 : 10,658
Unique customers     : 10,658
Duplicated customers : 0
Outstanding balance  : R$ 8,886,007.98

IN COLLECTIONS SINCE
min   2026-07-04
max   2026-09-30
Name: in_collections_since, dtype: datetime64[ns]

DPD ON 01/09
count   10,658.00
mean        15.60
std         19.34
min          0.00
25%          0.00
50%          4.00
75%         31.00
max         60.00
Name: days_past_due_on_2026-09-01, dtype: float64

✓ Queue reconciled: 10,658 unique customers


In [7]:
# ============================================================
# SEPTEMBER 2026 — MESSAGE PLAN
# STEP 2 — CHANNEL GUARDRAIL
#
# Exclude customers whose LAST observed WhatsApp status
# before September was:
#   - failed_invalid_number
#   - failed_blocked
#
# Expected:
# 10,658 -> 9,764 eligible
# ============================================================

wa_hist = wa.copy()

wa_hist["customer_id"] = wa_hist["customer_id"].astype(str)
wa_hist["sent_at"] = pd.to_datetime(wa_hist["sent_at"])

# ============================================================
# 1. ONLY INFORMATION AVAILABLE BEFORE SEPTEMBER
# ============================================================

wa_pre_sep = wa_hist.loc[
    wa_hist["sent_at"] < SEP01
].copy()

# ============================================================
# 2. LAST OBSERVED WHATSAPP CONTACT PER CUSTOMER
# ============================================================

last_wa = (
    wa_pre_sep
    .sort_values(["customer_id", "sent_at"])
    .groupby("customer_id", as_index=False)
    .tail(1)
    [
        [
            "customer_id",
            "sent_at",
            "delivery_status"
        ]
    ]
    .rename(
        columns={
            "sent_at": "last_wa_sent_at",
            "delivery_status": "last_delivery_status"
        }
    )
)

assert last_wa["customer_id"].is_unique

# ============================================================
# 3. MERGE WITH QUEUE
# ============================================================

base = base.merge(
    last_wa,
    on="customer_id",
    how="left",
    validate="1:1"
)

base["has_prior_wa_history"] = (
    base["last_wa_sent_at"].notna()
).astype(int)

BAD_STATUS = [
    "failed_invalid_number",
    "failed_blocked"
]

base["excluded_channel"] = (
    base["last_delivery_status"].isin(BAD_STATUS)
).astype(int)

# ============================================================
# 4. QA BEFORE FILTER
# ============================================================

print("=" * 90)
print("STEP 2 — CHANNEL GUARDRAIL")
print("=" * 90)

print(f"Queue customers       : {len(base):,}")
print(f"With prior WA history : {base['has_prior_wa_history'].sum():,}")
print(f"No prior WA history   : {(base['has_prior_wa_history'] == 0).sum():,}")

print("\nLAST DELIVERY STATUS")
print(
    base["last_delivery_status"]
    .fillna("NO_HISTORY")
    .value_counts()
    .to_string()
)

print("\n" + "-" * 90)
print("EXCLUSIONS")
print("-" * 90)

excluded = base.loc[
    base["excluded_channel"].eq(1)
].copy()

print(f"Invalid / blocked     : {len(excluded):,}")
print(
    f"Outstanding excluded  : "
    f"R$ {excluded['outstanding_balance_brl'].sum():,.2f}"
)

print("\nBREAKDOWN")
print(
    excluded["last_delivery_status"]
    .value_counts()
    .to_string()
)

# ============================================================
# 5. APPLY GUARDRAIL
# ============================================================

eligible = base.loc[
    base["excluded_channel"].eq(0)
].copy()

print("\n" + "-" * 90)
print("ELIGIBLE AFTER GUARDRAIL")
print("-" * 90)

print(f"Customers             : {len(eligible):,}")
print(f"Unique customers      : {eligible['customer_id'].nunique():,}")
print(
    f"Outstanding balance   : "
    f"R$ {eligible['outstanding_balance_brl'].sum():,.2f}"
)

# ============================================================
# 6. RECONCILIATION
# ============================================================

print("\n" + "-" * 90)
print("RECONCILIATION")
print("-" * 90)

print(f"Queue                  : {len(base):,}")
print(f"Excluded               : {len(excluded):,}")
print(f"Eligible               : {len(eligible):,}")
print(f"Excluded + eligible    : {len(excluded) + len(eligible):,}")

# ============================================================
# 7. HARD ASSERTIONS
# ============================================================

assert len(base) == 10_658
assert len(excluded) == 894, \
    f"Expected 894 excluded, got {len(excluded):,}"

assert len(eligible) == 9_764, \
    f"Expected 9,764 eligible, got {len(eligible):,}"

assert eligible["customer_id"].is_unique

assert len(excluded) + len(eligible) == len(base)

print("\n✓ Guardrail reconciled")
print("✓ 894 customers excluded")
print("✓ 9,764 customers eligible")

STEP 2 — CHANNEL GUARDRAIL
Queue customers       : 10,658
With prior WA history : 5,382
No prior WA history   : 5,276

LAST DELIVERY STATUS
last_delivery_status
NO_HISTORY               5276
delivered                4309
failed_blocked            490
failed_invalid_number     404
failed_unreachable        179

------------------------------------------------------------------------------------------
EXCLUSIONS
------------------------------------------------------------------------------------------
Invalid / blocked     : 894
Outstanding excluded  : R$ 764,423.24

BREAKDOWN
last_delivery_status
failed_blocked           490
failed_invalid_number    404

------------------------------------------------------------------------------------------
ELIGIBLE AFTER GUARDRAIL
------------------------------------------------------------------------------------------
Customers             : 9,764
Unique customers      : 9,764
Outstanding balance   : R$ 8,121,584.74

------------------------------

In [8]:
# ============================================================
# SEPTEMBER 2026 — MESSAGE PLAN
# STEP 3 — NEXT PAYDAY + DPD AT PAYDAY
#
# Expected:
# Eligible              = 9,764
# Payday in September   = 6,960
# Payday in October     = 2,804
# ============================================================

import calendar

plan = eligible.copy()

# ============================================================
# 1. PAYDAY DATE — SEPTEMBER / OCTOBER
# ============================================================

def safe_date(year, month, day):
    """
    Creates payday date safely.
    Example: payday=31 in a 30-day month -> last day of month.
    """
    last_day = calendar.monthrange(year, month)[1]
    day = min(int(day), last_day)

    return pd.Timestamp(
        year=year,
        month=month,
        day=day
    )


plan["payday_sep"] = plan["payday_day_of_month"].apply(
    lambda d: safe_date(2026, 9, d)
)

plan["payday_oct"] = plan["payday_day_of_month"].apply(
    lambda d: safe_date(2026, 10, d)
)

# ============================================================
# 2. NEXT PAYDAY AFTER ENTERING COLLECTIONS
#
# Important:
# equality is allowed.
#
# If customer enters collections exactly on payday,
# that day is DPD1 and payday is usable.
# ============================================================

plan["next_payday_date"] = np.where(
    plan["in_collections_since"] <= plan["payday_sep"],
    plan["payday_sep"],
    plan["payday_oct"]
)

plan["next_payday_date"] = pd.to_datetime(
    plan["next_payday_date"]
)

# ============================================================
# 3. PAYDAY MONTH
# ============================================================

plan["payday_month"] = np.where(
    plan["next_payday_date"].dt.month.eq(9),
    "September",
    "October"
)

# ============================================================
# 4. DPD AT NEXT PAYDAY
# ============================================================

plan["dpd_at_next_payday"] = (
    plan["next_payday_date"]
    - plan["in_collections_since"]
).dt.days + 1

# ============================================================
# 5. DPD BUCKET
# ============================================================

plan["dpd_payday_bucket"] = pd.cut(
    plan["dpd_at_next_payday"],
    bins=[0, 7, 15, 30, 45, 60, np.inf],
    labels=[
        "01–07",
        "08–15",
        "16–30",
        "31–45",
        "46–60",
        "60+"
    ],
    include_lowest=True
)

# ============================================================
# 6. QA — SEPTEMBER vs OCTOBER
# ============================================================

print("=" * 90)
print("STEP 3 — NEXT PAYDAY")
print("=" * 90)

month_summary = (
    plan.groupby("payday_month", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        outstanding_balance=("outstanding_balance_brl", "sum")
    )
)

print(month_summary.to_string())

# ============================================================
# 7. QA — DPD AT NEXT PAYDAY
# ============================================================

print("\n" + "-" * 90)
print("DPD AT NEXT PAYDAY")
print("-" * 90)

bucket_summary = (
    plan.groupby("dpd_payday_bucket", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        outstanding_balance=("outstanding_balance_brl", "sum")
    )
)

print(bucket_summary.to_string())

# ============================================================
# 8. CROSS — DPD × PAYDAY MONTH
# ============================================================

print("\n" + "-" * 90)
print("DPD BUCKET × PAYDAY MONTH")
print("-" * 90)

cross = pd.crosstab(
    plan["dpd_payday_bucket"],
    plan["payday_month"]
)

print(cross.to_string())

# ============================================================
# 9. RECONCILIATION
# ============================================================

n_sep = plan["payday_month"].eq("September").sum()
n_oct = plan["payday_month"].eq("October").sum()

print("\n" + "-" * 90)
print("RECONCILIATION")
print("-" * 90)

print(f"Eligible              : {len(plan):,}")
print(f"Next payday September : {n_sep:,}")
print(f"Next payday October   : {n_oct:,}")
print(f"Sep + Oct             : {n_sep + n_oct:,}")

# ============================================================
# 10. HARD ASSERTIONS
# ============================================================

assert len(plan) == 9_764

assert n_sep == 6_960, \
    f"Expected 6,960 September payday, got {n_sep:,}"

assert n_oct == 2_804, \
    f"Expected 2,804 October payday, got {n_oct:,}"

assert n_sep + n_oct == 9_764

print("\n✓ 9,764 eligible reconciled")
print("✓ 6,960 next payday in September")
print("✓ 2,804 next payday in October")

STEP 3 — NEXT PAYDAY
              customers  outstanding_balance
payday_month                                
October            2804         2,395,431.83
September          6960         5,726,152.91

------------------------------------------------------------------------------------------
DPD AT NEXT PAYDAY
------------------------------------------------------------------------------------------
                   customers  outstanding_balance
dpd_payday_bucket                                
01–07                   1281         1,083,953.37
08–15                   1882         1,604,068.73
16–30                   3492         2,969,226.39
31–45                   1221         1,001,894.61
46–60                   1103           867,203.22
60+                      785           595,238.42

------------------------------------------------------------------------------------------
DPD BUCKET × PAYDAY MONTH
-------------------------------------------------------------------------------

In [9]:
# ============================================================
# SEPTEMBER 2026 — MESSAGE PLAN
# STEP 4 — REBUILD CURRENT APPROVED RULES
# ============================================================

# ------------------------------------------------------------
# 1. PRIOR PAYMENT — information available before Sep/01
# ------------------------------------------------------------

wa_pre_sep = wa_hist.loc[
    wa_hist["sent_at"] < SEP01
].copy()

wa_pre_sep["payment_event"] = (
    pd.to_numeric(
        wa_pre_sep["amount_paid_brl"],
        errors="coerce"
    )
    .fillna(0)
    .gt(0)
    .astype(int)
)

prior_payment = (
    wa_pre_sep
    .groupby("customer_id", as_index=False)
    .agg(
        n_prior_payment_events=("payment_event", "sum")
    )
)

prior_payment["has_prior_payment"] = (
    prior_payment["n_prior_payment_events"] > 0
).astype(int)

plan = plan.merge(
    prior_payment[
        ["customer_id", "has_prior_payment"]
    ],
    on="customer_id",
    how="left",
    validate="1:1"
)

plan["has_prior_payment"] = (
    plan["has_prior_payment"]
    .fillna(0)
    .astype(int)
)

# ============================================================
# 2. INITIALIZE RULES
# ============================================================

plan["send_date"] = pd.NaT
plan["template"] = pd.NA
plan["rule"] = pd.NA

bucket = plan["dpd_payday_bucket"].astype(str)

# ============================================================
# 3. DPD 01–07
# Payday D+0
# Template will be randomized later
# ============================================================

m = (
    plan["payday_month"].eq("September")
    & bucket.eq("01–07")
)

plan.loc[m, "send_date"] = plan.loc[m, "next_payday_date"]
plan.loc[m, "rule"] = "01–07_D0"

# ============================================================
# 4. DPD 08–15
# Payday D+2
# ============================================================

m = (
    plan["payday_month"].eq("September")
    & bucket.eq("08–15")
)

plan.loc[m, "send_date"] = (
    plan.loc[m, "next_payday_date"]
    + pd.Timedelta(days=2)
)

plan.loc[m, "rule"] = "08–15_D2"

# ============================================================
# 5. DPD 16–30
# Payday D+0
# ============================================================

m = (
    plan["payday_month"].eq("September")
    & bucket.eq("16–30")
)

plan.loc[m, "send_date"] = plan.loc[m, "next_payday_date"]
plan.loc[m, "rule"] = "16–30_D0"

# ============================================================
# 6. DPD 31–45
# Payday D+0
#
# prior payment    -> Pix
# no prior payment -> Discount
# ============================================================

m31 = (
    plan["payday_month"].eq("September")
    & bucket.eq("31–45")
)

m = m31 & plan["has_prior_payment"].eq(1)

plan.loc[m, "send_date"] = plan.loc[m, "next_payday_date"]
plan.loc[m, "template"] = "pix_link"
plan.loc[m, "rule"] = "31–45_prior_pix"

m = m31 & plan["has_prior_payment"].eq(0)

plan.loc[m, "send_date"] = plan.loc[m, "next_payday_date"]
plan.loc[m, "template"] = "discount_offer"
plan.loc[m, "rule"] = "31–45_no_prior_discount"

# ============================================================
# 7. DPD 46–60
# Payday D+2
# ============================================================

m = (
    plan["payday_month"].eq("September")
    & bucket.eq("46–60")
)

plan.loc[m, "send_date"] = (
    plan.loc[m, "next_payday_date"]
    + pd.Timedelta(days=2)
)

plan.loc[m, "template"] = "urgent_reminder"
plan.loc[m, "rule"] = "46–60_D2_urgent"

# ============================================================
# 8. ONLY SEPTEMBER SENDS COUNT
# ============================================================

scheduled_sep = (
    plan["send_date"].between(SEP01, SEP30)
)

current_plan = plan.loc[
    scheduled_sep
].copy()

not_covered = plan.loc[
    ~scheduled_sep
].copy()

# ============================================================
# 9. QA
# ============================================================

print("=" * 90)
print("STEP 4 — CURRENT APPROVED SEPTEMBER RULES")
print("=" * 90)

print(f"Messages              : {len(current_plan):,}")
print(f"Unique customers      : {current_plan['customer_id'].nunique():,}")
print(
    f"Outstanding covered   : "
    f"R$ {current_plan['outstanding_balance_brl'].sum():,.2f}"
)

print("\nRULES")
print(
    current_plan["rule"]
    .value_counts()
    .to_string()
)

print("\n" + "-" * 90)
print("NOT COVERED")
print("-" * 90)

print(f"Customers             : {len(not_covered):,}")
print(
    f"Outstanding balance   : "
    f"R$ {not_covered['outstanding_balance_brl'].sum():,.2f}"
)

# ============================================================
# 10. REASON NOT COVERED
# ============================================================

not_covered["not_covered_reason"] = np.select(
    [
        not_covered["dpd_payday_bucket"].astype(str).eq("60+"),

        not_covered["payday_month"].eq("October"),

        not_covered["send_date"].gt(SEP30)
    ],
    [
        "DPD >60 at payday",
        "Next payday October",
        "D+2 spills into October"
    ],
    default="OTHER"
)

print("\nREASON")
print(
    not_covered["not_covered_reason"]
    .value_counts()
    .to_string()
)

# ============================================================
# 11. RECONCILIATION
# ============================================================

print("\n" + "-" * 90)
print("RECONCILIATION")
print("-" * 90)

print(f"Eligible              : {len(plan):,}")
print(f"Covered September     : {len(current_plan):,}")
print(f"Not covered           : {len(not_covered):,}")
print(
    f"Covered + not covered : "
    f"{len(current_plan) + len(not_covered):,}"
)

# ============================================================
# 12. HARD ASSERTIONS
# ============================================================

assert len(current_plan) == 5_914, \
    f"Expected 5,914, got {len(current_plan):,}"

assert len(not_covered) == 3_850, \
    f"Expected 3,850, got {len(not_covered):,}"

assert (
    len(current_plan) + len(not_covered)
    == 9_764
)

assert current_plan["customer_id"].is_unique
assert not_covered["customer_id"].is_unique

print("\n✓ 5,914 currently covered")
print("✓ 3,850 currently not covered")
print("✓ 9,764 fully reconciled")

STEP 4 — CURRENT APPROVED SEPTEMBER RULES
Messages              : 5,914
Unique customers      : 5,914
Outstanding covered   : R$ 4,921,718.67

RULES
rule
16–30_D0                   1583
01–07_D0                   1090
08–15_D2                   1033
31–45_no_prior_discount    1017
46–60_D2_urgent             987
31–45_prior_pix             204

------------------------------------------------------------------------------------------
NOT COVERED
------------------------------------------------------------------------------------------
Customers             : 3,850
Outstanding balance   : R$ 3,199,866.07

REASON
not_covered_reason
Next payday October        2804
DPD >60 at payday           785
D+2 spills into October     261

------------------------------------------------------------------------------------------
RECONCILIATION
------------------------------------------------------------------------------------------
Eligible              : 9,764
Covered September     : 5,914
Not cove

In [10]:
# ============================================================
# SEPTEMBER 2026 — MESSAGE PLAN
# STEP 5 — IDENTIFY THE 3,065 RANDOMIZATION CANDIDATES
#
# No randomization yet.
# ============================================================

# ============================================================
# 1. SELECT EXACTLY THE TWO GROUPS
# ============================================================

random_candidates = not_covered.loc[
    not_covered["not_covered_reason"].isin([
        "Next payday October",
        "D+2 spills into October"
    ])
].copy()

# ============================================================
# 2. DPD1–7 CALENDAR WINDOW
#
# DPD1 = in_collections_since
# DPD7 = in_collections_since + 6 days
# ============================================================

random_candidates["dpd1_date"] = (
    random_candidates["in_collections_since"]
)

random_candidates["dpd7_date"] = (
    random_candidates["in_collections_since"]
    + pd.Timedelta(days=6)
)

# ============================================================
# 3. INTERSECTION WITH SEPTEMBER
#
# Experimental window must satisfy BOTH:
#
#   DPD1–7
#   AND
#   September
# ============================================================

random_candidates["experiment_start"] = (
    random_candidates["dpd1_date"]
    .clip(lower=SEP01)
)

random_candidates["experiment_end"] = (
    random_candidates["dpd7_date"]
    .clip(upper=SEP30)
)

random_candidates["has_dpd1_7_window_sep"] = (
    random_candidates["experiment_start"]
    <= random_candidates["experiment_end"]
)

# ============================================================
# 4. NUMBER OF AVAILABLE DAYS
# ============================================================

random_candidates["n_available_days"] = np.where(
    random_candidates["has_dpd1_7_window_sep"],
    (
        random_candidates["experiment_end"]
        - random_candidates["experiment_start"]
    ).dt.days + 1,
    0
)

# ============================================================
# 5. DPD ON FIRST AVAILABLE SEPTEMBER DATE
# ============================================================

random_candidates["first_available_dpd"] = np.where(
    random_candidates["has_dpd1_7_window_sep"],
    (
        random_candidates["experiment_start"]
        - random_candidates["in_collections_since"]
    ).dt.days + 1,
    np.nan
)

random_candidates["last_available_dpd"] = np.where(
    random_candidates["has_dpd1_7_window_sep"],
    (
        random_candidates["experiment_end"]
        - random_candidates["in_collections_since"]
    ).dt.days + 1,
    np.nan
)

# ============================================================
# 6. QA
# ============================================================

print("=" * 90)
print("STEP 5 — RANDOMIZATION CANDIDATES")
print("=" * 90)

print(f"Candidates            : {len(random_candidates):,}")
print(
    f"Outstanding balance   : "
    f"R$ {random_candidates['outstanding_balance_brl'].sum():,.2f}"
)

print("\nSOURCE")
print(
    random_candidates["not_covered_reason"]
    .value_counts()
    .to_string()
)

print("\n" + "-" * 90)
print("DPD1–7 WINDOW AVAILABLE IN SEPTEMBER")
print("-" * 90)

print(
    random_candidates["has_dpd1_7_window_sep"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nNUMBER OF AVAILABLE DAYS")
print(
    random_candidates["n_available_days"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nFIRST AVAILABLE DPD")
print(
    random_candidates["first_available_dpd"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\nLAST AVAILABLE DPD")
print(
    random_candidates["last_available_dpd"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

# ============================================================
# 7. ENTRY DATE DISTRIBUTION
# ============================================================

print("\n" + "-" * 90)
print("IN COLLECTIONS SINCE")
print("-" * 90)

print(
    random_candidates["in_collections_since"]
    .agg(["min", "max"])
)

print("\nENTRY MONTH")
print(
    random_candidates["in_collections_since"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 8. HARD RECONCILIATION
# ============================================================

assert len(random_candidates) == 3_065, \
    f"Expected 3,065, got {len(random_candidates):,}"

assert (
    random_candidates["customer_id"].nunique()
    == 3_065
)

assert (
    random_candidates["not_covered_reason"]
    .eq("Next payday October")
    .sum()
    == 2_804
)

assert (
    random_candidates["not_covered_reason"]
    .eq("D+2 spills into October")
    .sum()
    == 261
)

print("\n✓ 3,065 candidates reconciled")
print("✓ 2,804 next payday October")
print("✓ 261 D+2 spills into October")

STEP 5 — RANDOMIZATION CANDIDATES
Candidates            : 3,065
Outstanding balance   : R$ 2,604,627.65

SOURCE
not_covered_reason
Next payday October        2804
D+2 spills into October     261

------------------------------------------------------------------------------------------
DPD1–7 WINDOW AVAILABLE IN SEPTEMBER
------------------------------------------------------------------------------------------
has_dpd1_7_window_sep
True     2949
False     116

NUMBER OF AVAILABLE DAYS
n_available_days
0     116
1     146
2     130
3     162
4     152
5     161
6     135
7    2063

FIRST AVAILABLE DPD
first_available_dpd
1.00    2949
NaN      116

LAST AVAILABLE DPD
last_available_dpd
1.00     146
2.00     130
3.00     162
4.00     152
5.00     161
6.00     135
7.00    2063
NaN      116

------------------------------------------------------------------------------------------
IN COLLECTIONS SINCE
-----------------------------------------------------------------------------------------

In [11]:
# ============================================================
# STEP 5B — 116 AUGUST CUSTOMERS
# MOVE TO LAYER 2 — ENGAGED RULE
# ============================================================

# ------------------------------------------------------------
# 1. Historical engagement BEFORE September
# ------------------------------------------------------------

wa_pre_sep = wa_hist.loc[
    wa_hist["sent_at"] < SEP01
].copy()

wa_pre_sep["engagement_event"] = (
    wa_pre_sep["interaction"]
    .fillna("none")
    .ne("none")
).astype(int)

prior_engagement = (
    wa_pre_sep
    .groupby("customer_id", as_index=False)
    .agg(
        has_prior_engagement=("engagement_event", "max")
    )
)

# ------------------------------------------------------------
# 2. Isolate the 116 August customers
# ------------------------------------------------------------

aug116 = random_candidates.loc[
    random_candidates["in_collections_since"].dt.month.eq(8)
].copy()

aug116 = aug116.merge(
    prior_engagement,
    on="customer_id",
    how="left",
    validate="1:1"
)

aug116["has_prior_engagement"] = (
    aug116["has_prior_engagement"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 3. DPD on September 1
# ------------------------------------------------------------

aug116["dpd_sep01"] = (
    SEP01 - aug116["in_collections_since"]
).dt.days + 1

# ------------------------------------------------------------
# 4. Layer 2 eligibility
# ------------------------------------------------------------

aug116["layer2_eligible"] = (
    aug116["has_prior_engagement"].eq(1)
    & aug116["dpd_sep01"].between(31, 60)
)

# ------------------------------------------------------------
# 5. Apply Layer 2
# ------------------------------------------------------------

aug116["layer2_send_date"] = pd.NaT
aug116["layer2_template"] = pd.NA
aug116["layer2_rule"] = pd.NA

# DPD 31–45 + engaged + prior payment -> PIX
m = (
    aug116["layer2_eligible"]
    & aug116["dpd_sep01"].between(31, 45)
    & aug116["has_prior_payment"].eq(1)
)

aug116.loc[m, "layer2_send_date"] = SEP01
aug116.loc[m, "layer2_template"] = "pix_link"
aug116.loc[m, "layer2_rule"] = "L2_31–45_prior_pix"

# DPD 31–45 + engaged + NO prior payment -> DISCOUNT
m = (
    aug116["layer2_eligible"]
    & aug116["dpd_sep01"].between(31, 45)
    & aug116["has_prior_payment"].eq(0)
)

aug116.loc[m, "layer2_send_date"] = SEP01
aug116.loc[m, "layer2_template"] = "discount_offer"
aug116.loc[m, "layer2_rule"] = "L2_31–45_no_prior_discount"

# DPD 46–60 + engaged -> URGENT
m = (
    aug116["layer2_eligible"]
    & aug116["dpd_sep01"].between(46, 60)
)

aug116.loc[m, "layer2_send_date"] = SEP01
aug116.loc[m, "layer2_template"] = "urgent_reminder"
aug116.loc[m, "layer2_rule"] = "L2_46–60_urgent"

# ============================================================
# 6. QA
# ============================================================

print("=" * 90)
print("116 AUGUST CUSTOMERS — LAYER 2")
print("=" * 90)

print(f"Total                         : {len(aug116):,}")
print(
    f"Engaged                       : "
    f"{aug116['has_prior_engagement'].sum():,}"
)
print(
    f"Not engaged                   : "
    f"{(aug116['has_prior_engagement'] == 0).sum():,}"
)

print("\nDPD ON 01/09")
print(
    pd.cut(
        aug116["dpd_sep01"],
        bins=[0,30,45,60,np.inf],
        labels=["01–30","31–45","46–60","60+"]
    )
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n" + "-" * 90)
print("LAYER 2 ASSIGNMENT")
print("-" * 90)

print(
    aug116["layer2_rule"]
    .fillna("NO CONTACT")
    .value_counts()
    .to_string()
)

covered_aug116 = aug116[
    "layer2_send_date"
].notna().sum()

not_covered_aug116 = len(aug116) - covered_aug116

print("\n" + "-" * 90)
print("RESULT")
print("-" * 90)

print(f"Covered by Layer 2            : {covered_aug116:,}")
print(f"Still without contact         : {not_covered_aug116:,}")
print(f"Total                         : {covered_aug116 + not_covered_aug116:,}")

assert len(aug116) == 116
assert aug116["customer_id"].is_unique
assert covered_aug116 + not_covered_aug116 == 116

print("\n✓ 116 reconciled")

116 AUGUST CUSTOMERS — LAYER 2
Total                         : 116
Engaged                       : 107
Not engaged                   : 9

DPD ON 01/09
dpd_sep01
01–30    107
31–45      9
46–60      0
60+        0

------------------------------------------------------------------------------------------
LAYER 2 ASSIGNMENT
------------------------------------------------------------------------------------------
layer2_rule
NO CONTACT                    107
L2_31–45_no_prior_discount      7
L2_31–45_prior_pix              2

------------------------------------------------------------------------------------------
RESULT
------------------------------------------------------------------------------------------
Covered by Layer 2            : 9
Still without contact         : 107
Total                         : 116

✓ 116 reconciled


In [12]:
# ============================================================
# STEP 6 — RANDOMIZE THE 2,804 OCTOBER-PAYDAY CUSTOMERS
#
# 1 message/customer
# send_date: randomized within available DPD1–7 in September
# template : 50/50 Friendly / Pix within assigned DPD
# ============================================================

import pandas as pd
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

# ============================================================
# 1. POPULATION — EXACTLY 2,804
# ============================================================

early2804 = random_candidates.loc[
    random_candidates["not_covered_reason"].eq("Next payday October")
].copy()

print("=" * 90)
print("STEP 6 — RANDOMIZATION POPULATION")
print("=" * 90)

print(f"Customers           : {len(early2804):,}")
print(f"Unique customers    : {early2804['customer_id'].nunique():,}")
print(
    f"Outstanding balance : "
    f"R$ {early2804['outstanding_balance_brl'].sum():,.2f}"
)

assert len(early2804) == 2_804
assert early2804["customer_id"].is_unique

# ============================================================
# 2. AVAILABLE DPDs IN SEPTEMBER
# ============================================================

def get_available_dpds(entry_date):

    available = []

    for dpd in range(1, 8):

        date = (
            entry_date
            + pd.Timedelta(days=dpd - 1)
        )

        if SEP01 <= date <= SEP30:
            available.append(dpd)

    return available


early2804["available_dpds"] = (
    early2804["in_collections_since"]
    .apply(get_available_dpds)
)

early2804["n_available_dpds"] = (
    early2804["available_dpds"].str.len()
)

# Every one of these 2,804 should have at least DPD1 available
assert early2804["n_available_dpds"].gt(0).all()

print("\nAVAILABLE DPDs")
print(
    early2804["n_available_dpds"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 3. BALANCED RANDOMIZATION OF DPD
#
# Customers with fewer options are allocated first.
# Among eligible DPDs, choose the currently least populated.
# Random tie-breaking.
# ============================================================

dpd_counts = {
    d: 0
    for d in range(1, 8)
}

# randomize first, then stable sort by constraint
allocation_order = (
    early2804[
        [
            "customer_id",
            "available_dpds",
            "n_available_dpds"
        ]
    ]
    .sample(frac=1, random_state=SEED)
    .sort_values(
        "n_available_dpds",
        kind="stable"
    )
)

assignments = []

for _, row in allocation_order.iterrows():

    eligible_dpds = row["available_dpds"]

    min_count = min(
        dpd_counts[d]
        for d in eligible_dpds
    )

    candidates = [
        d for d in eligible_dpds
        if dpd_counts[d] == min_count
    ]

    assigned_dpd = int(
        rng.choice(candidates)
    )

    dpd_counts[assigned_dpd] += 1

    assignments.append({
        "customer_id": row["customer_id"],
        "assigned_dpd": assigned_dpd
    })

assignments = pd.DataFrame(assignments)

early2804 = early2804.merge(
    assignments,
    on="customer_id",
    how="left",
    validate="1:1"
)

# ============================================================
# 4. SEND DATE
# ============================================================

early2804["send_date"] = (
    early2804["in_collections_since"]
    + pd.to_timedelta(
        early2804["assigned_dpd"] - 1,
        unit="D"
    )
)

# ============================================================
# 5. TEMPLATE — 50/50 WITHIN EACH DPD
# ============================================================

early2804["template"] = pd.NA

for dpd, idx in early2804.groupby(
    "assigned_dpd"
).groups.items():

    idx = np.array(list(idx))
    n = len(idx)

    n_friendly = n // 2
    n_pix = n - n_friendly

    templates = np.array(
        ["friendly_reminder"] * n_friendly
        + ["pix_link"] * n_pix
    )

    rng.shuffle(templates)

    early2804.loc[
        idx,
        "template"
    ] = templates

# ============================================================
# 6. CHECK ACTUAL DPD
# ============================================================

early2804["check_dpd"] = (
    early2804["send_date"]
    - early2804["in_collections_since"]
).dt.days + 1

# ============================================================
# 7. QA
# ============================================================

print("\n" + "=" * 90)
print("RANDOMIZATION RESULT")
print("=" * 90)

print(f"Messages             : {len(early2804):,}")
print(f"Unique customers     : {early2804['customer_id'].nunique():,}")

print("\nASSIGNED DPD")
print(
    early2804["assigned_dpd"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n" + "-" * 90)
print("TEMPLATE")
print("-" * 90)

print(
    early2804["template"]
    .value_counts()
    .to_string()
)

print("\nTEMPLATE × DPD")
print(
    pd.crosstab(
        early2804["assigned_dpd"],
        early2804["template"]
    ).to_string()
)

print("\n" + "-" * 90)
print("SEND DATE")
print("-" * 90)

print(
    early2804["send_date"]
    .agg(["min", "max"])
)

print("\nMESSAGES BY SEND DATE")

print(
    early2804["send_date"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 8. HARD ASSERTIONS
# ============================================================

assert len(early2804) == 2_804
assert early2804["customer_id"].is_unique

assert (
    early2804["assigned_dpd"]
    == early2804["check_dpd"]
).all()

assert early2804["assigned_dpd"].between(
    1, 7
).all()

assert early2804["send_date"].between(
    SEP01,
    SEP30
).all()

assert early2804["template"].isin(
    [
        "friendly_reminder",
        "pix_link"
    ]
).all()

assert (
    early2804.groupby("customer_id")
    .size()
    .eq(1)
    .all()
)

# Template balance
template_counts = early2804["template"].value_counts()

assert abs(
    template_counts.get("friendly_reminder", 0)
    - template_counts.get("pix_link", 0)
) <= 7  # at most one imbalance per DPD stratum

print("\n✓ 2,804 customers randomized")
print("✓ Exactly 1 message/customer")
print("✓ Every send occurs within customer's DPD1–7")
print("✓ Every send occurs in September")
print("✓ Friendly/Pix balanced ~50/50 within assigned DPD")

STEP 6 — RANDOMIZATION POPULATION
Customers           : 2,804
Unique customers    : 2,804
Outstanding balance : R$ 2,395,431.83

AVAILABLE DPDs
n_available_dpds
1     146
2     130
3     162
4     152
5     161
6     135
7    1918

RANDOMIZATION RESULT
Messages             : 2,804
Unique customers     : 2,804

ASSIGNED DPD
assigned_dpd
1    401
2    400
3    401
4    400
5    401
6    400
7    401

------------------------------------------------------------------------------------------
TEMPLATE
------------------------------------------------------------------------------------------
template
pix_link             1404
friendly_reminder    1400

TEMPLATE × DPD
template      friendly_reminder  pix_link
assigned_dpd                             
1                           200       201
2                           200       200
3                           200       201
4                           200       200
5                           200       201
6                           200     

In [13]:
# ============================================================
# QA — RANDOMIZATION INTEGRITY
# ============================================================

qa = early2804.copy()

qa["days_after_entry"] = (
    qa["send_date"] - qa["in_collections_since"]
).dt.days

print("=" * 90)
print("RANDOMIZATION INTEGRITY")
print("=" * 90)

print(
    f"Before collections entry : "
    f"{(qa['send_date'] < qa['in_collections_since']).sum():,}"
)

print(
    f"After DPD7              : "
    f"{(qa['days_after_entry'] > 6).sum():,}"
)

print(
    f"Outside September       : "
    f"{(~qa['send_date'].between(SEP01, SEP30)).sum():,}"
)

print(
    f"Duplicate customers     : "
    f"{qa['customer_id'].duplicated().sum():,}"
)

print("\nDPD DISTRIBUTION")
print(
    qa["assigned_dpd"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nTEMPLATE SHARE")
print(
    qa["template"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .astype(str)
    .add("%")
    .to_string()
)

assert (qa["send_date"] >= qa["in_collections_since"]).all()
assert qa["days_after_entry"].between(0, 6).all()
assert qa["send_date"].between(SEP01, SEP30).all()
assert qa["customer_id"].is_unique

print("\n✓ Randomization integrity validated")

RANDOMIZATION INTEGRITY
Before collections entry : 0
After DPD7              : 0
Outside September       : 0
Duplicate customers     : 0

DPD DISTRIBUTION
assigned_dpd
1    401
2    400
3    401
4    400
5    401
6    400
7    401

TEMPLATE SHARE
template
pix_link             50.07%
friendly_reminder    49.93%

✓ Randomization integrity validated


In [15]:
# ============================================================
# RECONCILIATION FROM THE PIECES ALREADY CREATED
# ============================================================

def ids(df):
    return set(df["customer_id"].astype(str))

eligible_ids = ids(eligible)

# ------------------------------------------------------------
# Known groups
# ------------------------------------------------------------

ids_current = ids(current_plan)
ids_2804    = ids(early2804)

aug_layer2 = aug116.loc[
    aug116["layer2_send_date"].notna()
].copy()

ids_aug_l2 = ids(aug_layer2)

# ------------------------------------------------------------
# Union
# ------------------------------------------------------------

known_ids = (
    ids_current
    | ids_2804
    | ids_aug_l2
)

missing_ids = eligible_ids - known_ids
unexpected_ids = known_ids - eligible_ids

print("=" * 100)
print("SEPTEMBER PLAN — RECONCILIATION FROM COMPONENTS")
print("=" * 100)

print(f"Eligible                         : {len(eligible_ids):,}")

print("\nKNOWN COMPONENTS")
print("-" * 100)

print(f"Current plan                     : {len(ids_current):,}")
print(f"October-payday randomized        : {len(ids_2804):,}")
print(f"August → Layer 2                 : {len(ids_aug_l2):,}")

print("\nOVERLAPS")
print("-" * 100)

print(
    f"Current ∩ 2804                   : "
    f"{len(ids_current & ids_2804):,}"
)

print(
    f"Current ∩ August L2              : "
    f"{len(ids_current & ids_aug_l2):,}"
)

print(
    f"2804 ∩ August L2                 : "
    f"{len(ids_2804 & ids_aug_l2):,}"
)

print("\nRECONCILIATION")
print("-" * 100)

print(f"Unique customers already assigned: {len(known_ids):,}")
print(f"Eligible still unassigned        : {len(missing_ids):,}")
print(f"Assigned but not eligible        : {len(unexpected_ids):,}")

# ============================================================
# PROFILE THE UNASSIGNED
# ============================================================

missing = eligible.loc[
    eligible["customer_id"].astype(str).isin(missing_ids)
].copy()

print("\n" + "=" * 100)
print("ELIGIBLE CUSTOMERS STILL UNASSIGNED")
print("=" * 100)

print(f"Customers           : {len(missing):,}")
print(
    f"Outstanding balance : "
    f"R$ {missing['outstanding_balance_brl'].sum():,.2f}"
)

print("\nDPD ON 01/09")
print("-" * 100)

print(
    pd.cut(
        missing["days_past_due_on_2026-09-01"],
        bins=[-1, 0, 7, 15, 30, 45, 60, np.inf],
        labels=[
            "DPD 0",
            "01–07",
            "08–15",
            "16–30",
            "31–45",
            "46–60",
            "60+"
        ]
    )
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nENTRY MONTH")
print("-" * 100)

print(
    missing["in_collections_since"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# CROSS WITH PREVIOUS NOT-COVERED GROUP
# ============================================================

print("\nOVERLAP WITH 116 AUGUST CUSTOMERS")
print("-" * 100)

print(
    f"Missing ∩ aug116 : "
    f"{len(missing_ids & ids(aug116)):,}"
)

print(
    f"Missing ∩ aug107 : "
    f"{len(missing_ids & ids(aug107)):,}"
)

# ============================================================
# TARGET
# ============================================================

print("\n" + "=" * 100)
print("TARGET CHECK")
print("=" * 100)

print(f"Eligible population       : {len(eligible_ids):,}")
print(f"Expected final plan       : 9,735")
print(f"Expected intentional skip : {len(eligible_ids) - 9_735:,}")

SEPTEMBER PLAN — RECONCILIATION FROM COMPONENTS
Eligible                         : 9,764

KNOWN COMPONENTS
----------------------------------------------------------------------------------------------------
Current plan                     : 5,914
October-payday randomized        : 2,804
August → Layer 2                 : 9

OVERLAPS
----------------------------------------------------------------------------------------------------
Current ∩ 2804                   : 0
Current ∩ August L2              : 0
2804 ∩ August L2                 : 0

RECONCILIATION
----------------------------------------------------------------------------------------------------
Unique customers already assigned: 8,727
Eligible still unassigned        : 1,037
Assigned but not eligible        : 0

ELIGIBLE CUSTOMERS STILL UNASSIGNED
Customers           : 1,037
Outstanding balance : R$ 799,445.13

DPD ON 01/09
----------------------------------------------------------------------------------------------------

NameError: name 'aug107' is not defined

In [16]:
# ============================================================
# STEP — DIAGNOSE THE 1,037 REMAINING CUSTOMERS
# ============================================================

remaining = missing.copy()

# ------------------------------------------------------------
# 1. PRIOR ENGAGEMENT — PIT before September
# ------------------------------------------------------------

wa_pre_sep = wa_hist.loc[
    wa_hist["sent_at"] < SEP01
].copy()

wa_pre_sep["engagement_event"] = (
    wa_pre_sep["interaction"]
    .fillna("none")
    .ne("none")
).astype(int)

prior_engagement = (
    wa_pre_sep
    .groupby("customer_id", as_index=False)
    .agg(
        has_prior_engagement=("engagement_event", "max")
    )
)

remaining = remaining.merge(
    prior_engagement,
    on="customer_id",
    how="left",
    validate="1:1"
)

remaining["has_prior_engagement"] = (
    remaining["has_prior_engagement"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 2. PRIOR PAYMENT
# ------------------------------------------------------------

remaining = remaining.merge(
    prior_payment[
        ["customer_id", "has_prior_payment"]
    ],
    on="customer_id",
    how="left",
    validate="1:1"
)

remaining["has_prior_payment"] = (
    remaining["has_prior_payment"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 3. SEGMENT
# ------------------------------------------------------------

remaining["segment"] = pd.cut(
    remaining["days_past_due_on_2026-09-01"],
    bins=[-1, 0, 7, 15, 30, 45, 60, np.inf],
    labels=[
        "DPD 0",
        "01–07",
        "08–15",
        "16–30",
        "31–45",
        "46–60",
        "60+"
    ]
)

# ------------------------------------------------------------
# 4. MASTER TABLE
# ------------------------------------------------------------

summary = (
    remaining
    .groupby(
        [
            "segment",
            "has_prior_engagement",
            "has_prior_payment"
        ],
        observed=True
    )
    .agg(
        customers=("customer_id", "nunique"),
        balance=("outstanding_balance_brl", "sum")
    )
    .reset_index()
)

print("=" * 100)
print("1,037 REMAINING CUSTOMERS — MASTER DIAGNOSTIC")
print("=" * 100)

print(summary.to_string(index=False))

# ------------------------------------------------------------
# 5. ENGAGEMENT BY DPD
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ENGAGEMENT × DPD")
print("=" * 100)

print(
    pd.crosstab(
        remaining["segment"],
        remaining["has_prior_engagement"],
        margins=True
    ).to_string()
)

# ------------------------------------------------------------
# 6. SPECIAL CHECK — 145 DPD0
# ------------------------------------------------------------

dpd0 = remaining.loc[
    remaining["segment"].eq("DPD 0")
].copy()

print("\n" + "=" * 100)
print("145 DPD0 — ENTRY DATES")
print("=" * 100)

print(f"Customers : {len(dpd0):,}")

print(
    dpd0["in_collections_since"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nEntry range:")
print(
    dpd0["in_collections_since"]
    .agg(["min", "max"])
)

# ------------------------------------------------------------
# 7. SPECIAL CHECK — 107 DPD16–30
# ------------------------------------------------------------

mid107 = remaining.loc[
    remaining["segment"].eq("16–30")
].copy()

print("\n" + "=" * 100)
print("107 DPD16–30")
print("=" * 100)

print(f"Customers : {len(mid107):,}")
print(
    f"Engaged   : "
    f"{mid107['has_prior_engagement'].sum():,}"
)
print(
    f"Not engaged: "
    f"{(mid107['has_prior_engagement'] == 0).sum():,}"
)

print("\nEntry dates:")
print(
    mid107["in_collections_since"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 8. LAYER 2 — 785
# ------------------------------------------------------------

late785 = remaining.loc[
    remaining["segment"].isin(["31–45", "46–60"])
].copy()

print("\n" + "=" * 100)
print("LAYER 2 POPULATION — 785")
print("=" * 100)

print(f"Customers    : {len(late785):,}")
print(
    f"Engaged      : "
    f"{late785['has_prior_engagement'].sum():,}"
)
print(
    f"Not engaged  : "
    f"{(late785['has_prior_engagement'] == 0).sum():,}"
)

print("\nBy DPD × engagement:")

print(
    pd.crosstab(
        late785["segment"],
        late785["has_prior_engagement"],
        margins=True
    ).to_string()
)

# ------------------------------------------------------------
# HARD RECONCILIATION
# ------------------------------------------------------------

assert len(remaining) == 1_037

assert (
    len(dpd0)
    + len(mid107)
    + len(late785)
    == 1_037
)

assert len(mid107) == 107
assert len(late785) == 785

print("\n✓ 1,037 completely reconciled")
print(
    f"✓ {len(dpd0):,} DPD0"
    f" + {len(mid107):,} DPD16–30"
    f" + {len(late785):,} DPD31–60"
    f" = {len(remaining):,}"
)

1,037 REMAINING CUSTOMERS — MASTER DIAGNOSTIC
segment  has_prior_engagement  has_prior_payment  customers    balance
  DPD 0                     0                  0        145 123,209.97
  16–30                     0                  0          9   6,116.94
  16–30                     1                  0         80  69,136.24
  16–30                     1                  1         18   5,743.56
  31–45                     0                  0          3   1,813.47
  31–45                     1                  0        129 107,563.84
  31–45                     1                  1         46  15,776.92
  46–60                     0                  0         17  12,887.01
  46–60                     1                  0        477 413,543.22
  46–60                     1                  1        113  43,653.96

ENGAGEMENT × DPD
has_prior_engagement    0    1   All
segment                             
DPD 0                 145    0   145
16–30                   9   98   107
31–45  

In [17]:
# ============================================================
# FINAL ELIGIBILITY DECISION — REMAINING 1,037
# ============================================================

remaining["include_final_plan"] = 0
remaining["final_reason"] = pd.NA

# ------------------------------------------------------------
# 1. NEW SEPTEMBER ENTRANTS
# No prior engagement expected/required
# ------------------------------------------------------------

m = remaining["segment"].eq("DPD 0")

remaining.loc[m, "include_final_plan"] = 1
remaining.loc[m, "final_reason"] = "new_sep_entrant"

# ------------------------------------------------------------
# 2. EXISTING CUSTOMERS — REQUIRE PRIOR ENGAGEMENT
# ------------------------------------------------------------

m = (
    remaining["segment"].isin(["16–30", "31–45", "46–60"])
    & remaining["has_prior_engagement"].eq(1)
)

remaining.loc[m, "include_final_plan"] = 1
remaining.loc[m, "final_reason"] = "layer2_engaged"

# ------------------------------------------------------------
# 3. EXISTING + NOT ENGAGED → NO CONTACT
# ------------------------------------------------------------

m = (
    remaining["segment"].isin(["16–30", "31–45", "46–60"])
    & remaining["has_prior_engagement"].eq(0)
)

remaining.loc[m, "final_reason"] = "no_contact_not_engaged"

# ============================================================
# QA
# ============================================================

remaining_in = remaining.loc[
    remaining["include_final_plan"].eq(1)
].copy()

remaining_out = remaining.loc[
    remaining["include_final_plan"].eq(0)
].copy()

print("=" * 90)
print("FINAL DECISION — REMAINING POPULATION")
print("=" * 90)

print(f"Remaining population : {len(remaining):,}")
print(f"Include in plan       : {len(remaining_in):,}")
print(f"No contact            : {len(remaining_out):,}")

print("\nINCLUDED")
print(
    pd.crosstab(
        remaining_in["segment"],
        remaining_in["has_prior_engagement"],
        margins=True
    ).to_string()
)

print("\nNO CONTACT")
print(
    pd.crosstab(
        remaining_out["segment"],
        remaining_out["has_prior_engagement"],
        margins=True
    ).to_string()
)

# ============================================================
# COMPLETE RECONCILIATION
# ============================================================

n_final = (
    len(ids_current)
    + len(ids_2804)
    + len(ids_aug_l2)
    + len(remaining_in)
)

print("\n" + "=" * 90)
print("FULL SEPTEMBER RECONCILIATION")
print("=" * 90)

print(f"Queue                     : {len(base):,}")
print(f"− invalid / blocked       : {len(excluded):,}")
print(f"= channel eligible        : {len(eligible):,}")
print(f"− not engaged / no contact: {len(remaining_out):,}")
print(f"= FINAL PLAN              : {n_final:,}")

assert len(remaining_in) == 1_008
assert len(remaining_out) == 29
assert n_final == 9_735

assert (
    remaining_out["has_prior_engagement"].eq(0)
).all()

print("\n✓ 1,008 remaining customers included")
print("✓ 29 intentionally receive no message")
print("✓ FINAL SEPTEMBER POPULATION = 9,735")

FINAL DECISION — REMAINING POPULATION
Remaining population : 1,037
Include in plan       : 1,008
No contact            : 29

INCLUDED
has_prior_engagement    0    1   All
segment                             
DPD 0                 145    0   145
16–30                   0   98    98
31–45                   0  175   175
46–60                   0  590   590
All                   145  863  1008

NO CONTACT
has_prior_engagement   0  All
segment                      
16–30                  9    9
31–45                  3    3
46–60                 17   17
All                   29   29

FULL SEPTEMBER RECONCILIATION
Queue                     : 10,658
− invalid / blocked       : 894
= channel eligible        : 9,764
− not engaged / no contact: 29
= FINAL PLAN              : 9,735

✓ 1,008 remaining customers included
✓ 29 intentionally receive no message
✓ FINAL SEPTEMBER POPULATION = 9,735


In [18]:
# ============================================================
# AUDIT — WHERE DO THE 1,008 COME FROM?
# DO NOT ASSIGN ANY NEW RULE
# ============================================================

audit1008 = remaining_in.copy()

print("=" * 100)
print("1,008 INCLUDED CUSTOMERS — EXISTING ASSIGNMENTS AUDIT")
print("=" * 100)

print(f"Customers        : {len(audit1008):,}")
print(f"Unique customers : {audit1008['customer_id'].nunique():,}")

# ============================================================
# 1. BASIC SEGMENTS
# ============================================================

print("\nSEGMENT")
print("-" * 100)

print(
    audit1008["segment"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 2. WHAT DATE/RULE COLUMNS ALREADY EXIST?
# ============================================================

possible_cols = [
    "send_date",
    "rule",
    "next_payday_date",
    "dpd_at_next_payday",
    "dpd_payday_bucket",
    "layer2_send_date",
    "layer2_template",
    "layer2_rule",
    "template"
]

existing_cols = [
    c for c in possible_cols
    if c in audit1008.columns
]

print("\nEXISTING STRATEGY COLUMNS")
print("-" * 100)

print(existing_cols)

for col in existing_cols:
    print(
        f"{col:<25} "
        f"non-null={audit1008[col].notna().sum():>5,} | "
        f"null={audit1008[col].isna().sum():>5,}"
    )

# ============================================================
# 3. ORIGINAL SEND_DATE BY SEGMENT
# ============================================================

if "send_date" in audit1008.columns:

    print("\nORIGINAL send_date × SEGMENT")
    print("-" * 100)

    tmp = (
        audit1008
        .assign(has_send_date=audit1008["send_date"].notna())
        .groupby(
            ["segment", "has_send_date"],
            observed=True
        )
        .size()
        .unstack(fill_value=0)
    )

    print(tmp.to_string())

# ============================================================
# 4. ORIGINAL RULE BY SEGMENT
# ============================================================

if "rule" in audit1008.columns:

    print("\nORIGINAL RULE × SEGMENT")
    print("-" * 100)

    print(
        pd.crosstab(
            audit1008["segment"],
            audit1008["rule"].fillna("NO_RULE")
        ).to_string()
    )

# ============================================================
# 5. NEXT PAYDAY / ORIGINAL INTENDED SEND
# ============================================================

cols = [
    "customer_id",
    "segment",
    "in_collections_since",
    "days_past_due_on_2026-09-01",
    "next_payday_date",
    "dpd_at_next_payday",
    "dpd_payday_bucket",
    "rule",
    "send_date"
]

cols = [c for c in cols if c in audit1008.columns]

print("\nSAMPLE / CUSTOMER LEVEL")
print("-" * 100)

display(
    audit1008[cols]
    .sort_values(
        ["segment", "in_collections_since", "customer_id"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 6. CHECK AGAINST ALL OBJECTS WE HAVE ALREADY CREATED
# ============================================================

def overlap_df(label, df):
    if df is None:
        return

    x = set(audit1008["customer_id"].astype(str))
    y = set(df["customer_id"].astype(str))

    print(f"{label:<35}: {len(x & y):>5,}")


print("\nOVERLAP WITH EXISTING STRATEGY OBJECTS")
print("-" * 100)

overlap_df("current_plan", current_plan)
overlap_df("early2804", early2804)
overlap_df("aug116", aug116)
overlap_df("aug_layer2", aug_layer2)

# ============================================================
# 7. EXACT BREAKDOWN OF THE 1,008
# ============================================================

print("\n" + "=" * 100)
print("EXACT BREAKDOWN")
print("=" * 100)

breakdown = (
    audit1008
    .groupby(
        [
            "segment",
            "has_prior_engagement",
            "has_prior_payment"
        ],
        observed=True
    )
    .agg(
        customers=("customer_id", "nunique"),
        balance=("outstanding_balance_brl", "sum")
    )
    .reset_index()
)

print(breakdown.to_string(index=False))

assert len(audit1008) == 1_008
assert audit1008["customer_id"].is_unique

print("\n✓ 1,008 audited — no new assignment made")

1,008 INCLUDED CUSTOMERS — EXISTING ASSIGNMENTS AUDIT
Customers        : 1,008
Unique customers : 1,008

SEGMENT
----------------------------------------------------------------------------------------------------
segment
DPD 0    145
01–07      0
08–15      0
16–30     98
31–45    175
46–60    590
60+        0

EXISTING STRATEGY COLUMNS
----------------------------------------------------------------------------------------------------
[]

SAMPLE / CUSTOMER LEVEL
----------------------------------------------------------------------------------------------------


,customer_id,segment,in_collections_since,days_past_due_on_2026-09-01
0,C012389,DPD 0,2026-09-16,0
1,C012792,DPD 0,2026-09-16,0
2,C013293,DPD 0,2026-09-16,0
3,C013356,DPD 0,2026-09-16,0
4,C013415,DPD 0,2026-09-16,0
...,...,...,...,...
1003,C008740,46–60,2026-07-18,46
1004,C009328,46–60,2026-07-18,46
1005,C009532,46–60,2026-07-18,46
1006,C009555,46–60,2026-07-18,46



OVERLAP WITH EXISTING STRATEGY OBJECTS
----------------------------------------------------------------------------------------------------
current_plan                       :     0
early2804                          :     0
aug116                             :    98
aug_layer2                         :     0

EXACT BREAKDOWN
segment  has_prior_engagement  has_prior_payment  customers    balance
  DPD 0                     0                  0        145 123,209.97
  16–30                     1                  0         80  69,136.24
  16–30                     1                  1         18   5,743.56
  31–45                     1                  0        129 107,563.84
  31–45                     1                  1         46  15,776.92
  46–60                     1                  0        477 413,543.22
  46–60                     1                  1        113  43,653.96

✓ 1,008 audited — no new assignment made


In [19]:
# ============================================================
# FINAL PLAN — MATERIALIZE THE 9,735 CUSTOMERS
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. CAMADA 1 — 5,914
# ============================================================

layer1 = current_plan.copy()

assert len(layer1) == 5_914
assert layer1["customer_id"].nunique() == 5_914

print("=" * 90)
print("LAYER 1")
print("=" * 90)
print(f"Customers              : {len(layer1):,}")
print(f"send_date non-null     : {layer1['send_date'].notna().sum():,}")
print(f"send_date missing      : {layer1['send_date'].isna().sum():,}")


# ============================================================
# 2. CAMADA 2 — 765 ENGAGED
# ============================================================

layer2 = late785.loc[
    late785["has_prior_engagement"].eq(1)
].copy()

assert len(layer2) == 765
assert layer2["customer_id"].nunique() == 765

print("\n" + "=" * 90)
print("LAYER 2")
print("=" * 90)
print(f"Customers              : {len(layer2):,}")

# Show what operational columns actually exist
print(
    "Date columns available:",
    [c for c in layer2.columns if "date" in c.lower()]
)

print(
    "Template columns available:",
    [c for c in layer2.columns if "template" in c.lower()]
)


# ============================================================
# 3. AUGUST ENGAGED — 107
# ============================================================

aug107_engaged = aug116.loc[
    aug116["has_prior_engagement"].eq(1)
].copy()

assert len(aug107_engaged) == 107
assert aug107_engaged["customer_id"].nunique() == 107

print("\n" + "=" * 90)
print("AUGUST ENGAGED")
print("=" * 90)
print(f"Customers              : {len(aug107_engaged):,}")

print(
    "Date columns available:",
    [c for c in aug107_engaged.columns if "date" in c.lower()]
)

print(
    "Template columns available:",
    [c for c in aug107_engaged.columns if "template" in c.lower()]
)


# ============================================================
# 4. DPD1–7 RANDOMIZED — 2,949
# ============================================================

# 2,804 already randomized
rand2804 = early2804.copy()

# The remaining 145 are the September entrants from remaining_in
new145 = remaining_in.loc[
    remaining_in["segment"].eq("DPD 0")
].copy()

print("\n" + "=" * 90)
print("DPD1–7 RANDOMIZATION")
print("=" * 90)

print(f"Already randomized     : {len(rand2804):,}")
print(f"Additional Sep entrants: {len(new145):,}")
print(f"Expected total         : {len(rand2804) + len(new145):,}")

assert len(rand2804) == 2_804
assert len(new145) == 145
assert len(rand2804) + len(new145) == 2_949


# ============================================================
# 5. CURRENT STATUS OF SEND_DATE
# ============================================================

pieces = {
    "Layer 1": layer1,
    "Layer 2": layer2,
    "August engaged": aug107_engaged,
    "Randomized 2,804": rand2804,
    "New September 145": new145
}

print("\n" + "=" * 90)
print("SEND_DATE STATUS BEFORE CONCATENATION")
print("=" * 90)

status = []

for name, x in pieces.items():

    has_col = "send_date" in x.columns

    non_null = (
        x["send_date"].notna().sum()
        if has_col else 0
    )

    status.append({
        "component": name,
        "customers": len(x),
        "has_send_date_column": has_col,
        "send_date_non_null": non_null,
        "send_date_missing": len(x) - non_null
    })

status = pd.DataFrame(status)

print(status.to_string(index=False))


# ============================================================
# 6. CUSTOMER-ID RECONCILIATION
# ============================================================

all_ids = pd.concat(
    [
        layer1[["customer_id"]],
        layer2[["customer_id"]],
        aug107_engaged[["customer_id"]],
        rand2804[["customer_id"]],
        new145[["customer_id"]]
    ],
    ignore_index=True
)

print("\n" + "=" * 90)
print("CUSTOMER RECONCILIATION")
print("=" * 90)

print(f"Rows assembled        : {len(all_ids):,}")
print(f"Unique customers      : {all_ids['customer_id'].nunique():,}")
print(f"Duplicated customers  : {all_ids['customer_id'].duplicated().sum():,}")

assert len(all_ids) == 9_735
assert all_ids["customer_id"].nunique() == 9_735

print("\n✓ Population = 9,735")

LAYER 1
Customers              : 5,914
send_date non-null     : 5,914
send_date missing      : 0

LAYER 2
Customers              : 765
Date columns available: []
Template columns available: []

AUGUST ENGAGED
Customers              : 107
Date columns available: ['next_payday_date', 'send_date', 'dpd1_date', 'dpd7_date', 'layer2_send_date']
Template columns available: ['template', 'layer2_template']

DPD1–7 RANDOMIZATION
Already randomized     : 2,804
Additional Sep entrants: 145
Expected total         : 2,949

SEND_DATE STATUS BEFORE CONCATENATION
        component  customers  has_send_date_column  send_date_non_null  send_date_missing
          Layer 1       5914                  True                5914                  0
          Layer 2        765                 False                   0                765
   August engaged        107                  True                 107                  0
 Randomized 2,804       2804                  True                2804                

In [21]:
# ============================================================
# CHECKPOINT — OBJECTS READY FOR FINAL_PLAN
# ============================================================

objects = {
    "current_plan": current_plan,
    "early2804": early2804,
    "aug116": aug116,
    "late785": late785,
}

print("=" * 90)
print("OBJECTS AVAILABLE FOR FINAL PLAN")
print("=" * 90)

for name, x in objects.items():

    print(f"\n{name}")
    print("-" * 60)

    print(f"Rows             : {len(x):,}")
    print(f"Unique customers : {x['customer_id'].nunique():,}")

    for col in [
        "send_date",
        "layer2_send_date",
        "template",
        "layer2_template"
    ]:
        if col in x.columns:
            print(
                f"{col:<20}: "
                f"{x[col].notna().sum():,} non-null"
            )

OBJECTS AVAILABLE FOR FINAL PLAN

current_plan
------------------------------------------------------------
Rows             : 5,914
Unique customers : 5,914
send_date           : 5,914 non-null
template            : 2,208 non-null

early2804
------------------------------------------------------------
Rows             : 2,804
Unique customers : 2,804
send_date           : 2,804 non-null
template            : 2,804 non-null

aug116
------------------------------------------------------------
Rows             : 116
Unique customers : 116
send_date           : 116 non-null
layer2_send_date    : 9 non-null
template            : 116 non-null
layer2_template     : 9 non-null

late785
------------------------------------------------------------
Rows             : 785
Unique customers : 785


In [22]:
# ============================================================
# COMPLETE TEMPLATE — LAYER 1
# Missing 3,706 = Early/Mid 50/50 Friendly/Pix
# Keep send_date completely unchanged
# ============================================================

rng_l1 = np.random.default_rng(42)

layer1 = current_plan.copy()

send_date_original = layer1["send_date"].copy()

print("=" * 90)
print("LAYER 1 — TEMPLATE COMPLETION")
print("=" * 90)

print(f"Customers        : {len(layer1):,}")
print(f"send_date filled : {layer1['send_date'].notna().sum():,}")
print(f"template filled  : {layer1['template'].notna().sum():,}")
print(f"template missing : {layer1['template'].isna().sum():,}")

# Only rows whose template is still missing
missing_template = layer1["template"].isna()

print("\nRULES WITH MISSING TEMPLATE")
print(
    layer1.loc[missing_template, "rule"]
    .value_counts()
    .to_string()
)

# These MUST be exactly the 3 early/mid rules
expected_rules = {
    "01–07_D0",
    "08–15_D2",
    "16–30_D0"
}

observed_rules = set(
    layer1.loc[missing_template, "rule"].dropna().unique()
)

assert observed_rules == expected_rules
assert missing_template.sum() == 3_706

# ============================================================
# 50/50 WITHIN EACH RULE
# ============================================================

for rule, idx in (
    layer1.loc[missing_template]
    .groupby("rule")
    .groups
    .items()
):

    idx = np.array(list(idx))
    n = len(idx)

    templates = np.array(
        ["friendly_reminder"] * (n // 2)
        + ["pix_link"] * (n - n // 2)
    )

    rng_l1.shuffle(templates)

    layer1.loc[idx, "template"] = templates

# ============================================================
# QA
# ============================================================

print("\n" + "-" * 90)
print("FINAL LAYER 1")
print("-" * 90)

print(f"Customers        : {len(layer1):,}")
print(f"send_date filled : {layer1['send_date'].notna().sum():,}")
print(f"template filled  : {layer1['template'].notna().sum():,}")

print("\nTEMPLATE × RULE")

print(
    pd.crosstab(
        layer1["rule"],
        layer1["template"]
    ).to_string()
)

assert len(layer1) == 5_914
assert layer1["customer_id"].is_unique
assert layer1["send_date"].notna().all()
assert layer1["template"].notna().all()

# Critical: send_date must remain untouched
assert (
    pd.to_datetime(layer1["send_date"])
    .reset_index(drop=True)
    .equals(
        pd.to_datetime(send_date_original)
        .reset_index(drop=True)
    )
)

print("\n✓ Layer 1 = 5,914")
print("✓ send_date = 5,914")
print("✓ template  = 5,914")
print("✓ Existing send_date unchanged")

LAYER 1 — TEMPLATE COMPLETION
Customers        : 5,914
send_date filled : 5,914
template filled  : 2,208
template missing : 3,706

RULES WITH MISSING TEMPLATE
rule
16–30_D0    1583
01–07_D0    1090
08–15_D2    1033

------------------------------------------------------------------------------------------
FINAL LAYER 1
------------------------------------------------------------------------------------------
Customers        : 5,914
send_date filled : 5,914
template filled  : 5,914

TEMPLATE × RULE
template                 discount_offer  friendly_reminder  pix_link  urgent_reminder
rule                                                                                 
01–07_D0                              0                545       545                0
08–15_D2                              0                516       517                0
16–30_D0                              0                791       792                0
31–45_no_prior_discount            1017                  0        

In [23]:
# ============================================================
# LAYER 2 — INSPECT 765 ENGAGED
# ============================================================

layer2_765 = late785.loc[
    late785["has_prior_engagement"].eq(1)
].copy()

print("=" * 90)
print("LAYER 2 — 765 ENGAGED")
print("=" * 90)

print(f"Rows             : {len(layer2_765):,}")
print(f"Unique customers : {layer2_765['customer_id'].nunique():,}")

print("\nCOLUMNS:")
print(layer2_765.columns.tolist())

print("\nSEGMENT:")
print(layer2_765["segment"].value_counts(dropna=False).to_string())

print("\nPRIOR PAYMENT:")
for c in layer2_765.columns:
    if "prior" in c.lower() or "payment" in c.lower():
        print(c, ":", layer2_765[c].value_counts(dropna=False).head(10).to_dict())

assert len(layer2_765) == 765
assert layer2_765["customer_id"].nunique() == 765

LAYER 2 — 765 ENGAGED
Rows             : 765
Unique customers : 765

COLUMNS:
['customer_id', 'in_collections_since', 'days_past_due_on_2026-09-01', 'outstanding_balance_brl', 'monthly_salary_brl', 'payday_day_of_month', 'n_prior_transactions', 'account_age_months', 'days_since_last_app_login', 'state_uf', 'last_wa_sent_at', 'last_delivery_status', 'has_prior_wa_history', 'excluded_channel', 'has_prior_engagement', 'has_prior_payment', 'segment']

SEGMENT:
segment
46–60    590
31–45    175
DPD 0      0
08–15      0
01–07      0
16–30      0
60+        0

PRIOR PAYMENT:
n_prior_transactions : {1: 103, 3: 86, 2: 76, 4: 59, 5: 57, 6: 48, 8: 42, 11: 34, 7: 33, 9: 24}
has_prior_wa_history : {1: 765}
has_prior_engagement : {1: 765}
has_prior_payment : {0: 606, 1: 159}


In [24]:
# ============================================================
# LAYER 2 — MATERIALIZE 765 ENGAGED CUSTOMERS
# ============================================================

layer2 = layer2_765.copy()

# ------------------------------------------------------------
# SEND DATE
# ------------------------------------------------------------

layer2["send_date"] = pd.Timestamp("2026-09-01")

# ------------------------------------------------------------
# TEMPLATE
# ------------------------------------------------------------

layer2["template"] = np.nan

# 31–45 + prior payment -> Pix
mask = (
    layer2["segment"].eq("31–45")
    & layer2["has_prior_payment"].eq(1)
)

layer2.loc[mask, "template"] = "pix_link"

# 31–45 + no prior payment -> Discount
mask = (
    layer2["segment"].eq("31–45")
    & layer2["has_prior_payment"].eq(0)
)

layer2.loc[mask, "template"] = "discount_offer"

# 46–60 -> Urgent
mask = layer2["segment"].eq("46–60")

layer2.loc[mask, "template"] = "urgent_reminder"


# ============================================================
# QA
# ============================================================

print("=" * 90)
print("LAYER 2 — FINAL")
print("=" * 90)

print(f"Customers        : {len(layer2):,}")
print(f"Unique customers : {layer2['customer_id'].nunique():,}")
print(f"send_date filled : {layer2['send_date'].notna().sum():,}")
print(f"template filled  : {layer2['template'].notna().sum():,}")

print("\nSEGMENT × PRIOR PAYMENT × TEMPLATE")
print("-" * 90)

qa = (
    layer2
    .groupby(
        ["segment", "has_prior_payment", "template"],
        observed=True
    )
    .size()
    .rename("customers")
)

print(qa.to_string())


# ============================================================
# HARD ASSERTIONS
# ============================================================

assert len(layer2) == 765
assert layer2["customer_id"].nunique() == 765
assert layer2["customer_id"].is_unique

assert layer2["send_date"].notna().all()
assert layer2["template"].notna().all()

assert layer2["send_date"].eq(
    pd.Timestamp("2026-09-01")
).all()

assert (
    layer2.loc[
        layer2["segment"].eq("31–45")
        & layer2["has_prior_payment"].eq(1),
        "template"
    ].eq("pix_link")
).all()

assert (
    layer2.loc[
        layer2["segment"].eq("31–45")
        & layer2["has_prior_payment"].eq(0),
        "template"
    ].eq("discount_offer")
).all()

assert (
    layer2.loc[
        layer2["segment"].eq("46–60"),
        "template"
    ].eq("urgent_reminder")
).all()

print("\n✓ Layer 2 = 765")
print("✓ send_date = 765")
print("✓ template  = 765")

LAYER 2 — FINAL
Customers        : 765
Unique customers : 765
send_date filled : 765
template filled  : 765

SEGMENT × PRIOR PAYMENT × TEMPLATE
------------------------------------------------------------------------------------------
segment  has_prior_payment  template       
31–45    0                  discount_offer     129
         1                  pix_link            46
46–60    0                  urgent_reminder    477
         1                  urgent_reminder    113

✓ Layer 2 = 765
✓ send_date = 765
✓ template  = 765


In [26]:
# ============================================================
# AUGUST 116 — AUDIT
# Reconstruct segment from DPD on 2026-09-01
# ============================================================

aug = aug116.copy()

# ------------------------------------------------------------
# 1. RECONSTRUCT DPD SEGMENT
# ------------------------------------------------------------

aug["segment"] = pd.cut(
    aug["days_past_due_on_2026-09-01"],
    bins=[-np.inf, 0, 7, 15, 30, 45, 60, np.inf],
    labels=[
        "DPD 0",
        "01–07",
        "08–15",
        "16–30",
        "31–45",
        "46–60",
        "60+"
    ]
)

# ------------------------------------------------------------
# 2. SPLIT ENGAGED / NON-ENGAGED
# ------------------------------------------------------------

aug107 = aug.loc[
    aug["has_prior_engagement"].eq(1)
].copy()

aug9_no_contact = aug.loc[
    aug["has_prior_engagement"].eq(0)
].copy()


# ============================================================
# 3. QA
# ============================================================

print("=" * 90)
print("AUGUST 116 — AUDIT")
print("=" * 90)

print(f"Customers        : {len(aug):,}")
print(f"Unique customers : {aug['customer_id'].nunique():,}")

print("\nENGAGEMENT")
print("-" * 90)

print(
    aug["has_prior_engagement"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nSEGMENT × ENGAGEMENT")
print("-" * 90)

print(
    pd.crosstab(
        aug["segment"],
        aug["has_prior_engagement"],
        observed=True
    ).to_string()
)

print("\nENGAGED ONLY — SEGMENT × PRIOR PAYMENT")
print("-" * 90)

print(
    pd.crosstab(
        aug107["segment"],
        aug107["has_prior_payment"],
        observed=True
    ).to_string()
)

print("\nENGAGED — EXISTING SEND_DATE")
print("-" * 90)

print(
    pd.to_datetime(aug107["send_date"])
    .dt.strftime("%Y-%m-%d")
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\nENGAGED — EXISTING TEMPLATE")
print("-" * 90)

print(
    aug107["template"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nNON-ENGAGED")
print("-" * 90)

print(f"Customers : {len(aug9_no_contact):,}")

print(
    aug9_no_contact["segment"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# 4. HARD ASSERTIONS
# ============================================================

assert len(aug) == 116
assert aug["customer_id"].nunique() == 116

assert len(aug107) == 107
assert aug107["customer_id"].nunique() == 107

assert len(aug9_no_contact) == 9
assert aug9_no_contact["customer_id"].nunique() == 9

print("\n✓ August total       = 116")
print("✓ Engaged / contact = 107")
print("✓ No contact        = 9")

AUGUST 116 — AUDIT
Customers        : 116
Unique customers : 116

ENGAGEMENT
------------------------------------------------------------------------------------------
has_prior_engagement
0      9
1    107

SEGMENT × ENGAGEMENT
------------------------------------------------------------------------------------------


TypeError: crosstab() got an unexpected keyword argument 'observed'

In [27]:
# ============================================================
# CONTINUE AUGUST 116 AUDIT
# ============================================================

# Objects already created:
# aug
# aug107
# aug9_no_contact

print("\nSEGMENT × ENGAGEMENT")
print("-" * 90)

print(
    pd.crosstab(
        aug["segment"],
        aug["has_prior_engagement"]
    ).to_string()
)


print("\nENGAGED ONLY — SEGMENT × PRIOR PAYMENT")
print("-" * 90)

print(
    pd.crosstab(
        aug107["segment"],
        aug107["has_prior_payment"]
    ).to_string()
)


print("\nENGAGED — EXISTING SEND_DATE")
print("-" * 90)

print(
    pd.to_datetime(aug107["send_date"])
    .dt.strftime("%Y-%m-%d")
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)


print("\nENGAGED — EXISTING TEMPLATE")
print("-" * 90)

print(
    aug107["template"]
    .value_counts(dropna=False)
    .to_string()
)


print("\nNON-ENGAGED — SEGMENT")
print("-" * 90)

print(
    aug9_no_contact["segment"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# ASSERTIONS
# ============================================================

assert len(aug) == 116
assert len(aug107) == 107
assert len(aug9_no_contact) == 9

assert aug107["customer_id"].nunique() == 107
assert aug9_no_contact["customer_id"].nunique() == 9

print("\n" + "=" * 90)
print("RECONCILIATION")
print("=" * 90)
print(f"August total       : {len(aug):,}")
print(f"Engaged / contact  : {len(aug107):,}")
print(f"No contact         : {len(aug9_no_contact):,}")
print(f"Check              : {len(aug107) + len(aug9_no_contact):,}")

print("\n✓ 107 + 9 = 116")


SEGMENT × ENGAGEMENT
------------------------------------------------------------------------------------------
has_prior_engagement  0   1
segment                    
16–30                 9  98
31–45                 0   9

ENGAGED ONLY — SEGMENT × PRIOR PAYMENT
------------------------------------------------------------------------------------------
has_prior_payment   0   1
segment                  
16–30              80  18
31–45               7   2

ENGAGED — EXISTING SEND_DATE
------------------------------------------------------------------------------------------
send_date
2026-10-02    107

ENGAGED — EXISTING TEMPLATE
------------------------------------------------------------------------------------------
template
urgent_reminder    107

NON-ENGAGED — SEGMENT
------------------------------------------------------------------------------------------
segment
16–30    9
01–07    0
DPD 0    0
08–15    0
31–45    0
46–60    0
60+      0

RECONCILIATION
August total       : 116

In [28]:
# ============================================================
# AUGUST ENGAGED — MATERIALIZE 107
# ============================================================

rng_aug = np.random.default_rng(42)

aug107_final = aug107.copy()

# All are contacted immediately in September
aug107_final["send_date"] = pd.Timestamp("2026-09-01")
aug107_final["template"] = pd.NA


# ============================================================
# 1. DPD 16–30 — 98 ENGAGED
#    50/50 Friendly / Pix
# ============================================================

mask_1630 = aug107_final["segment"].eq("16–30")

idx = aug107_final.index[mask_1630].to_numpy()

assert len(idx) == 98

templates = np.array(
    ["friendly_reminder"] * 49 +
    ["pix_link"] * 49
)

rng_aug.shuffle(templates)

aug107_final.loc[idx, "template"] = templates


# ============================================================
# 2. DPD 31–45 — 9 ENGAGED
#
# prior payment    -> Pix
# no prior payment -> Discount
# ============================================================

mask_3145_prior = (
    aug107_final["segment"].eq("31–45")
    & aug107_final["has_prior_payment"].eq(1)
)

mask_3145_no_prior = (
    aug107_final["segment"].eq("31–45")
    & aug107_final["has_prior_payment"].eq(0)
)

aug107_final.loc[
    mask_3145_prior,
    "template"
] = "pix_link"

aug107_final.loc[
    mask_3145_no_prior,
    "template"
] = "discount_offer"


# ============================================================
# QA
# ============================================================

print("=" * 90)
print("AUGUST ENGAGED — FINAL")
print("=" * 90)

print(f"Customers        : {len(aug107_final):,}")
print(f"Unique customers : {aug107_final['customer_id'].nunique():,}")
print(f"send_date filled : {aug107_final['send_date'].notna().sum():,}")
print(f"template filled  : {aug107_final['template'].notna().sum():,}")

print("\nSEGMENT × TEMPLATE")
print("-" * 90)

print(
    pd.crosstab(
        aug107_final["segment"],
        aug107_final["template"]
    ).to_string()
)

print("\nSEND DATE")
print("-" * 90)

print(
    aug107_final["send_date"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# HARD ASSERTIONS
# ============================================================

assert len(aug107_final) == 107
assert aug107_final["customer_id"].nunique() == 107
assert aug107_final["customer_id"].is_unique

assert aug107_final["send_date"].notna().all()
assert aug107_final["template"].notna().all()

assert aug107_final["send_date"].eq(
    pd.Timestamp("2026-09-01")
).all()

# 16–30
assert (
    aug107_final.loc[mask_1630, "template"]
    .value_counts()
    .to_dict()
    ==
    {
        "friendly_reminder": 49,
        "pix_link": 49
    }
)

# 31–45
assert mask_3145_no_prior.sum() == 7
assert mask_3145_prior.sum() == 2

assert aug107_final.loc[
    mask_3145_no_prior, "template"
].eq("discount_offer").all()

assert aug107_final.loc[
    mask_3145_prior, "template"
].eq("pix_link").all()

print("\n✓ August engaged = 107")
print("✓ DPD 16–30 = 98 → 49 Friendly / 49 Pix")
print("✓ DPD 31–45 = 9 → 7 Discount / 2 Pix")
print("✓ send_date = 2026-09-01 for all 107")
print("✓ 9 non-engaged remain NO CONTACT")

AUGUST ENGAGED — FINAL
Customers        : 107
Unique customers : 107
send_date filled : 107
template filled  : 107

SEGMENT × TEMPLATE
------------------------------------------------------------------------------------------
template  discount_offer  friendly_reminder  pix_link
segment                                              
16–30                  0                 49        49
31–45                  7                  0         2

SEND DATE
------------------------------------------------------------------------------------------
send_date
2026-09-01    107

✓ August engaged = 107
✓ DPD 16–30 = 98 → 49 Friendly / 49 Pix
✓ DPD 31–45 = 9 → 7 Discount / 2 Pix
✓ send_date = 2026-09-01 for all 107
✓ 9 non-engaged remain NO CONTACT


In [29]:
# ============================================================
# FINAL RANDOMIZATION BLOCK
# 2,804 FROZEN + 145 NEW = 2,949
# ============================================================

rng145 = np.random.default_rng(2026)

# ------------------------------------------------------------
# 1. GET THE 145 SEPTEMBER ENTRANTS
# ------------------------------------------------------------

new145 = remaining_in.loc[
    remaining_in["segment"].eq("DPD 0")
].copy()

assert len(new145) == 145
assert new145["customer_id"].nunique() == 145

new145["in_collections_since"] = pd.to_datetime(
    new145["in_collections_since"]
)

print("=" * 90)
print("NEW SEPTEMBER ENTRANTS")
print("=" * 90)

print(f"Customers : {len(new145):,}")
print(
    f"Entry     : "
    f"{new145['in_collections_since'].min().date()} "
    f"to {new145['in_collections_since'].max().date()}"
)


# ------------------------------------------------------------
# 2. RANDOMIZE DPD 1–7
#
# 145 / 7:
# five DPDs receive 21 customers
# two DPDs receive 20
# ------------------------------------------------------------

dpds = np.arange(1, 8)

base = len(new145) // 7       # 20
remainder = len(new145) % 7   # 5

# Randomly choose which 5 DPDs receive the extra customer
extra_dpds = rng145.choice(
    dpds,
    size=remainder,
    replace=False
)

dpd_pool = np.repeat(dpds, base)

dpd_pool = np.concatenate([
    dpd_pool,
    extra_dpds
])

rng145.shuffle(dpd_pool)

new145["assigned_dpd"] = dpd_pool.astype(int)


# ------------------------------------------------------------
# 3. SEND DATE
# DPD1 = entry date
# DPD7 = entry + 6 days
# ------------------------------------------------------------

new145["send_date"] = (
    new145["in_collections_since"]
    + pd.to_timedelta(
        new145["assigned_dpd"] - 1,
        unit="D"
    )
)


# ------------------------------------------------------------
# 4. TEMPLATE — BALANCED FRIENDLY / PIX
#
# 145 is odd:
# one template gets 73, the other 72
# Which one gets the extra is randomized.
# ------------------------------------------------------------

templates = np.array(
    ["friendly_reminder"] * 72 +
    ["pix_link"] * 72
)

extra_template = rng145.choice(
    ["friendly_reminder", "pix_link"]
)

templates = np.concatenate([
    templates,
    [extra_template]
])

rng145.shuffle(templates)

new145["template"] = templates


# ============================================================
# 5. QA — 145
# ============================================================

print("\n" + "=" * 90)
print("145 RANDOMIZATION — QA")
print("=" * 90)

print("\nASSIGNED DPD")
print(
    new145["assigned_dpd"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nTEMPLATE")
print(
    new145["template"]
    .value_counts()
    .to_string()
)

print("\nSEND DATE")
print(
    f"{new145['send_date'].min().date()} "
    f"to {new145['send_date'].max().date()}"
)

assert len(new145) == 145
assert new145["customer_id"].is_unique
assert new145["assigned_dpd"].between(1, 7).all()

assert new145["send_date"].dt.month.eq(9).all()
assert new145["send_date"].dt.year.eq(2026).all()

# Check date corresponds exactly to assigned DPD
calculated_dpd = (
    new145["send_date"]
    - new145["in_collections_since"]
).dt.days + 1

assert calculated_dpd.equals(
    new145["assigned_dpd"]
)


# ============================================================
# 6. FREEZE 2,804 EXISTING RANDOMIZATION
# ============================================================

rand2804 = early2804.copy()

assert len(rand2804) == 2_804
assert rand2804["customer_id"].nunique() == 2_804
assert rand2804["send_date"].notna().all()
assert rand2804["template"].notna().all()


# ============================================================
# 7. RANDOMIZED POPULATION = 2,949
# ============================================================

random2949 = pd.concat(
    [
        rand2804,
        new145
    ],
    ignore_index=True,
    sort=False
)

assert len(random2949) == 2_949
assert random2949["customer_id"].nunique() == 2_949
assert random2949["customer_id"].is_unique
assert random2949["send_date"].notna().all()
assert random2949["template"].notna().all()

print("\n" + "=" * 90)
print("RANDOMIZED POPULATION")
print("=" * 90)

print(f"Existing frozen : {len(rand2804):,}")
print(f"New randomized  : {len(new145):,}")
print(f"TOTAL           : {len(random2949):,}")

print("\n✓ Randomized population = 2,949")

NEW SEPTEMBER ENTRANTS
Customers : 145
Entry     : 2026-09-16 to 2026-09-23

145 RANDOMIZATION — QA

ASSIGNED DPD
assigned_dpd
1    21
2    20
3    21
4    21
5    21
6    20
7    21

TEMPLATE
template
friendly_reminder    73
pix_link             72

SEND DATE
2026-09-16 to 2026-09-29

RANDOMIZED POPULATION
Existing frozen : 2,804
New randomized  : 145
TOTAL           : 2,949

✓ Randomized population = 2,949


In [30]:
# ============================================================
# FINAL SEPTEMBER PLAN — 9,735 CUSTOMERS
# ============================================================

final_plan = pd.concat(
    [
        layer1[["customer_id", "send_date", "template"]],
        layer2[["customer_id", "send_date", "template"]],
        aug107_final[["customer_id", "send_date", "template"]],
        random2949[["customer_id", "send_date", "template"]],
    ],
    ignore_index=True
)

# Normalize date
final_plan["send_date"] = pd.to_datetime(final_plan["send_date"])

# ============================================================
# QA
# ============================================================

print("=" * 90)
print("FINAL SEPTEMBER PLAN")
print("=" * 90)

print(f"Rows              : {len(final_plan):,}")
print(f"Unique customers  : {final_plan['customer_id'].nunique():,}")
print(f"send_date COUNT   : {final_plan['send_date'].count():,}")
print(f"template COUNT    : {final_plan['template'].count():,}")
print(f"Duplicate IDs     : {final_plan['customer_id'].duplicated().sum():,}")
print(f"Missing send_date : {final_plan['send_date'].isna().sum():,}")
print(f"Missing template  : {final_plan['template'].isna().sum():,}")

print("\nDATE RANGE")
print(f"Min : {final_plan['send_date'].min().date()}")
print(f"Max : {final_plan['send_date'].max().date()}")

print("\n" + "-" * 90)
print("MESSAGES BY DATE")
print("-" * 90)

print(
    final_plan["send_date"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# HARD ASSERTIONS
# ============================================================

assert len(final_plan) == 9_735
assert final_plan["customer_id"].nunique() == 9_735
assert final_plan["customer_id"].is_unique

assert final_plan["send_date"].count() == 9_735
assert final_plan["template"].count() == 9_735

assert final_plan["send_date"].dt.year.eq(2026).all()
assert final_plan["send_date"].dt.month.eq(9).all()

print("\n✓ FINAL PLAN       = 9,735")
print("✓ UNIQUE CUSTOMERS = 9,735")
print("✓ SEND_DATE COUNT  = 9,735")
print("✓ TEMPLATE COUNT   = 9,735")
print("✓ ALL DATES IN SEPTEMBER")

FINAL SEPTEMBER PLAN
Rows              : 9,735
Unique customers  : 9,735
send_date COUNT   : 9,735
template COUNT    : 9,735
Duplicate IDs     : 0
Missing send_date : 0
Missing template  : 0

DATE RANGE
Min : 2026-09-01
Max : 2026-09-30

------------------------------------------------------------------------------------------
MESSAGES BY DATE
------------------------------------------------------------------------------------------
send_date
2026-09-01    1112
2026-09-02       1
2026-09-03     141
2026-09-04       7
2026-09-05    1211
2026-09-06      19
2026-09-07     721
2026-09-08      27
2026-09-09      41
2026-09-10     498
2026-09-11      71
2026-09-12     347
2026-09-13      53
2026-09-14      83
2026-09-15     453
2026-09-16      97
2026-09-17     404
2026-09-18     104
2026-09-19     114
2026-09-20     744
2026-09-21     129
2026-09-22     499
2026-09-23     119
2026-09-24     156
2026-09-25     582
2026-09-26     101
2026-09-27     339
2026-09-28      72
2026-09-29      69
20

In [31]:
# ============================================================
# RANDOMIZE SEND HOUR — ALL 9,735 CUSTOMERS
# Hours: 09–20
# Balanced randomized allocation
# ============================================================

import numpy as np

HOUR_SEED = 2026
rng_hour = np.random.default_rng(HOUR_SEED)

# Freeze current decisions for QA
dates_before = final_plan["send_date"].copy()
templates_before = final_plan["template"].copy()

# ============================================================
# 1. BUILD BALANCED HOUR POOL
# ============================================================

hours = np.arange(9, 21)   # 9, 10, ..., 20

n = len(final_plan)

base = n // len(hours)       # 811
remainder = n % len(hours)   # 3

# Every hour gets 811
hour_pool = np.repeat(hours, base)

# Randomly select 3 hours to receive one extra customer
extra_hours = rng_hour.choice(
    hours,
    size=remainder,
    replace=False
)

hour_pool = np.concatenate([
    hour_pool,
    extra_hours
])

# Randomize assignment across customers
rng_hour.shuffle(hour_pool)

final_plan["send_hour"] = hour_pool.astype(int)


# ============================================================
# 2. QA — DISTRIBUTION
# ============================================================

print("=" * 90)
print("SEND HOUR RANDOMIZATION — 9,735 CUSTOMERS")
print("=" * 90)

hour_qa = (
    final_plan["send_hour"]
    .value_counts()
    .sort_index()
    .rename("customers")
    .to_frame()
)

hour_qa["share_pct"] = (
    hour_qa["customers"]
    / len(final_plan)
    * 100
).round(3)

print(hour_qa.to_string())

print("\nHours receiving +1:")
print(sorted(extra_hours.tolist()))


# ============================================================
# 3. INTEGRITY
# ============================================================

print("\n" + "-" * 90)
print("INTEGRITY")
print("-" * 90)

print(f"Rows              : {len(final_plan):,}")
print(f"Unique customers  : {final_plan['customer_id'].nunique():,}")
print(f"send_date COUNT   : {final_plan['send_date'].count():,}")
print(f"send_hour COUNT   : {final_plan['send_hour'].count():,}")
print(f"template COUNT    : {final_plan['template'].count():,}")
print(f"Min hour          : {final_plan['send_hour'].min()}")
print(f"Max hour          : {final_plan['send_hour'].max()}")

# ============================================================
# 4. HARD ASSERTIONS
# ============================================================

assert len(final_plan) == 9_735
assert final_plan["customer_id"].nunique() == 9_735
assert final_plan["customer_id"].is_unique

assert final_plan["send_hour"].notna().all()
assert final_plan["send_hour"].between(9, 20).all()

counts = final_plan["send_hour"].value_counts()

assert len(counts) == 12
assert counts.min() == 811
assert counts.max() == 812

# Make sure NOTHING ELSE changed
assert final_plan["send_date"].equals(dates_before)
assert final_plan["template"].equals(templates_before)

print("\n✓ 9,735 customers randomized")
print("✓ All hours between 09 and 20")
print("✓ Every hour has 811 or 812 customers")
print("✓ send_date unchanged")
print("✓ template unchanged")


# ============================================================
# 5. FINAL OUTPUT — EXACT CASE FORMAT
# ============================================================

final_output = final_plan[
    [
        "customer_id",
        "send_date",
        "send_hour",
        "template"
    ]
].copy()

# Required date format: YYYY-MM-DD
final_output["send_date"] = (
    final_output["send_date"]
    .dt.strftime("%Y-%m-%d")
)

# Required integer hour
final_output["send_hour"] = (
    final_output["send_hour"]
    .astype(int)
)

print("\n" + "=" * 90)
print("FINAL OUTPUT")
print("=" * 90)

print(final_output.head(20).to_string(index=False))

print("\nShape:", final_output.shape)

SEND HOUR RANDOMIZATION — 9,735 CUSTOMERS
           customers  share_pct
send_hour                      
9                812       8.34
10               812       8.34
11               811       8.33
12               811       8.33
13               811       8.33
14               811       8.33
15               811       8.33
16               811       8.33
17               812       8.34
18               811       8.33
19               811       8.33
20               811       8.33

Hours receiving +1:
[9, 10, 17]

------------------------------------------------------------------------------------------
INTEGRITY
------------------------------------------------------------------------------------------
Rows              : 9,735
Unique customers  : 9,735
send_date COUNT   : 9,735
send_hour COUNT   : 9,735
template COUNT    : 9,735
Min hour          : 9
Max hour          : 20

✓ 9,735 customers randomized
✓ All hours between 09 and 20
✓ Every hour has 811 or 812 customers
✓ send_date

In [32]:
# ============================================================
# EXPORT FINAL SEPTEMBER PLAN
# ============================================================

OUTPUT_PATH = "../data/processed/september_2026_message_plan.csv"

final_output.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows : {len(final_output):,}")
print(f"Cols : {final_output.columns.tolist()}")

assert len(final_output) == 9_735
assert final_output["customer_id"].is_unique
assert final_output.isna().sum().sum() == 0

print("\n✓ FINAL SEPTEMBER MESSAGE PLAN FROZEN")

Saved: ../data/processed/september_2026_message_plan.csv
Rows : 9,735
Cols : ['customer_id', 'send_date', 'send_hour', 'template']

✓ FINAL SEPTEMBER MESSAGE PLAN FROZEN


In [34]:
# ============================================================
# UPDATE FINAL PLAN — SEND_HOUR ONLY
# ============================================================
#
# Objetivo:
#   - manter o plan.csv já validado
#   - NÃO reconstruir nenhuma regra
#   - alterar SOMENTE send_hour
#   - horários permitidos: 18h e 19h
#   - distribuição global ~50/50
#   - estratificação: send_date × template
#
# Como temos 9.735 clientes:
#   um horário terá 4.868
#   outro terá 4.867
#
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 0. CONFIG
# ============================================================

PATH = r"../data/processed/plan.csv"

SEED = 2026


# ============================================================
# 1. LOAD FROZEN PLAN
# ============================================================

plan = pd.read_csv(PATH)

plan["send_date"] = pd.to_datetime(plan["send_date"])


print("=" * 100)
print("ORIGINAL PLAN")
print("=" * 100)

print(f"Rows             : {len(plan):,}")
print(f"Unique customers : {plan['customer_id'].nunique():,}")
print(f"Duplicates       : {plan['customer_id'].duplicated().sum():,}")

print("\nColumns:")
print(plan.columns.tolist())

print("\nCurrent send_hour distribution:")

print(
    plan["send_hour"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 2. HARD VALIDATION BEFORE CHANGING ANYTHING
# ============================================================

assert len(plan) == 9735, (
    f"Expected 9,735 rows, found {len(plan):,}"
)

assert plan["customer_id"].nunique() == 9735, (
    "customer_id is not unique."
)

assert plan["customer_id"].duplicated().sum() == 0, (
    "Duplicated customers found."
)

required_columns = {
    "customer_id",
    "send_date",
    "send_hour",
    "template"
}

missing = required_columns - set(plan.columns)

assert len(missing) == 0, (
    f"Missing columns: {missing}"
)


print("\n✓ Original plan structure validated")


# ============================================================
# 3. FREEZE ORIGINAL PLAN
# ============================================================

# We will use this copy to prove that NOTHING
# except send_hour changed.

before = plan.copy(deep=True)


# ============================================================
# 4. RANDOM NUMBER GENERATOR
# ============================================================

rng = np.random.default_rng(SEED)


# ============================================================
# 5. CREATE NEW SEND_HOUR
# ============================================================

new_hour = pd.Series(
    index=plan.index,
    dtype="Int64"
)


# ============================================================
# 6. STRATIFIED RANDOMIZATION
#
# STRATA:
# send_date × template
#
# Within every stratum:
#
#      ~50% → 18h
#      ~50% → 19h
#
# For odd-sized strata, one observation necessarily remains.
# The extra observation is randomly assigned to 18 or 19.
# ============================================================

groups = plan.groupby(
    ["send_date", "template"],
    observed=True
).groups


for _, idx in groups.items():

    idx = np.array(list(idx))

    # Randomize customers inside the stratum
    idx = rng.permutation(idx)

    n = len(idx)

    # Base allocation
    n18 = n // 2
    n19 = n // 2

    # Odd stratum:
    # randomly decide where the extra customer goes
    if n % 2 == 1:

        if rng.integers(0, 2) == 0:
            n18 += 1
        else:
            n19 += 1

    # Assign hours
    new_hour.loc[idx[:n18]] = 18
    new_hour.loc[idx[n18:]] = 19


# ============================================================
# 7. UPDATE ONLY SEND_HOUR
# ============================================================

assert new_hour.notna().all(), (
    "Some customers did not receive a send_hour."
)

plan["send_hour"] = new_hour.astype(int)


# ============================================================
# 8. BASIC QA
# ============================================================

print("\n" + "=" * 100)
print("NEW SEND_HOUR DISTRIBUTION")
print("=" * 100)

hour_counts = (
    plan["send_hour"]
    .value_counts()
    .sort_index()
)

print(hour_counts)


print("\nPercent:")

print(
    (
        plan["send_hour"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(3)
)


# ============================================================
# 9. PROVE ONLY SEND_HOUR CHANGED
# ============================================================

protected_cols = [
    col
    for col in before.columns
    if col != "send_hour"
]


print("\n" + "=" * 100)
print("COLUMN INTEGRITY CHECK")
print("=" * 100)


for col in protected_cols:

    original = (
        before[col]
        .reset_index(drop=True)
    )

    updated = (
        plan[col]
        .reset_index(drop=True)
    )

    assert original.equals(updated), (
        f"ERROR: column '{col}' changed!"
    )

    print(f"✓ {col}: unchanged")


print("\n✓ ONLY send_hour was modified")


# ============================================================
# 10. STRUCTURAL QA
# ============================================================

assert len(plan) == 9735

assert plan["customer_id"].nunique() == 9735

assert plan["customer_id"].duplicated().sum() == 0

assert set(plan["send_hour"].unique()) == {18, 19}


n18 = (plan["send_hour"] == 18).sum()
n19 = (plan["send_hour"] == 19).sum()


print("\n" + "=" * 100)
print("GLOBAL BALANCE")
print("=" * 100)

print(f"18h : {n18:,}")
print(f"19h : {n19:,}")
print(f"Diff: {abs(n18 - n19):,}")


# ============================================================
# 11. STRATIFICATION QA
# ============================================================

strata = (
    plan
    .groupby(
        ["send_date", "template", "send_hour"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
)


# Guarantee both hour columns exist
for h in [18, 19]:

    if h not in strata.columns:
        strata[h] = 0


strata["difference"] = (
    strata[18] - strata[19]
).abs()


print("\n" + "=" * 100)
print("STRATIFICATION QA — SEND_DATE × TEMPLATE")
print("=" * 100)

print(f"Number of strata    : {len(strata):,}")
print(f"Maximum imbalance   : {strata['difference'].max():,}")

print("\nImbalance distribution:")

print(
    strata["difference"]
    .value_counts()
    .sort_index()
)


assert strata["difference"].max() <= 1, (
    "A send_date × template stratum is not balanced."
)


print(
    "\n✓ Every send_date × template stratum "
    "is balanced 18h/19h with maximum difference = 1"
)


# ============================================================
# 12. TEMPLATE × HOUR
# ============================================================

print("\n" + "=" * 100)
print("TEMPLATE × SEND HOUR")
print("=" * 100)

template_hour = pd.crosstab(
    plan["template"],
    plan["send_hour"],
    margins=True
)

print(template_hour)


# ============================================================
# 13. SEND DATE × HOUR
# ============================================================

print("\n" + "=" * 100)
print("SEND DATE × SEND HOUR")
print("=" * 100)

date_hour = pd.crosstab(
    plan["send_date"],
    plan["send_hour"],
    margins=True
)

print(date_hour)


# ============================================================
# 14. FINAL RECONCILIATION
# ============================================================

print("\n" + "=" * 100)
print("FINAL RECONCILIATION")
print("=" * 100)

print(f"Customers total : {len(plan):,}")
print(f"18h             : {n18:,}")
print(f"19h             : {n19:,}")
print(f"18h + 19h       : {n18 + n19:,}")

assert n18 + n19 == 9735


# ============================================================
# 15. RESTORE SEND_DATE FORMAT
# ============================================================

plan["send_date"] = (
    plan["send_date"]
    .dt.strftime("%Y-%m-%d")
)


# ============================================================
# 16. PRESERVE EXACT ORIGINAL COLUMN ORDER
# ============================================================

plan = plan[before.columns]


# ============================================================
# 17. EXPORT
# ============================================================

plan.to_csv(
    PATH,
    index=False
)


print("\n" + "=" * 100)
print("EXPORT COMPLETE")
print("=" * 100)

print(f"Saved to: {PATH}")

print(
    "\n✓ Original plan preserved\n"
    "✓ customer_id unchanged\n"
    "✓ send_date unchanged\n"
    "✓ template unchanged\n"
    "✓ ONLY send_hour changed\n"
    "✓ send_hour restricted to 18h / 19h\n"
    "✓ stratified by send_date × template"
)

ORIGINAL PLAN
Rows             : 9,735
Unique customers : 9,735
Duplicates       : 0

Columns:
['customer_id', 'send_date', 'send_hour', 'template']

Current send_hour distribution:
send_hour
9     812
10    812
11    811
12    811
13    811
14    811
15    811
16    811
17    812
18    811
19    811
20    811
Name: count, dtype: int64

✓ Original plan structure validated

NEW SEND_HOUR DISTRIBUTION
send_hour
18    4869
19    4866
Name: count, dtype: int64

Percent:
send_hour
18   50.02
19   49.98
Name: proportion, dtype: float64

COLUMN INTEGRITY CHECK
✓ customer_id: unchanged
✓ send_date: unchanged
✓ template: unchanged

✓ ONLY send_hour was modified

GLOBAL BALANCE
18h : 4,869
19h : 4,866
Diff: 3

STRATIFICATION QA — SEND_DATE × TEMPLATE
Number of strata    : 73
Maximum imbalance   : 1

Imbalance distribution:
difference
0    36
1    37
Name: count, dtype: int64

✓ Every send_date × template stratum is balanced 18h/19h with maximum difference = 1

TEMPLATE × SEND HOUR
send_hour     